# Lateral profile analysis

Cleaned workflow for loading measured lateral profile data, processing simulation files, edge-aligning profiles, and producing measured-vs-simulation plots with percent-difference and gamma summaries.

Run the notebook from top to bottom after updating the settings cell for the applicator, Excel sheet, measured columns, and simulation files you want to analyze.


## 1. Imports and plotting defaults


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['legend.fontsize'] = 10



## 2. Reference measured PDD data


In [ ]:
MEASURED_PDD = {
    (10, 'flash9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            94.97,94.97,95.15,97.66,98.18,98.66,99.09,99.35,99.52,99.57,
            99.78,99.91,99.96,100.00,99.91,99.78,99.52,99.31,98.87,98.79,
            97.83,97.40,96.62,95.58,94.54,93.33,91.68,90.21,88.04,85.83,
            81.46,74.96,68.02,59.27,50.69,41.33,32.06,23.27,14.99,8.15,
            5.59,0.35,0.52,0.52
        ], dtype=float),
    },
    (10, 'conv9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            94.62,94.39,96.64,96.86,97.31,98.65,97.76,98.88,99.10,99.10,
            98.65,99.10,99.10,100.00,98.65,99.33,98.43,98.21,97.31,97.53,
            96.19,96.19,94.39,94.39,92.60,91.03,90.36,87.89,85.87,83.18,
            80.27,74.22,66.82,58.97,51.57,40.81,32.06,23.54,15.92,9.64,
            5.38,1.12,1.12,1.12
        ], dtype=float),
    },
    (10, 'flash6alt'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            92.59,94.51,95.91,97.13,98.15,98.72,99.36,99.84,100.00,99.96,
            99.57,99.04,98.02,96.42,94.32,92.08,88.89,85.44,81.48,76.76,
            71.52,66.28,60.34,54.79,48.72,43.04,37.04,31.35,25.93,20.75,
            16.09,8.49,3.83,1.53,0.64,0.38,0.38,0.38,0.38,0.38,
            0.38,0.38,0.38,0.38
        ], dtype=float),
    },
    (5, 'flash9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            93.22,93.74,94.99,95.84,96.41,97.04,97.61,97.84,98.52,98.80,
            99.32,99.43,99.69,99.83,99.83,100.00,99.91,99.60,99.43,98.97,
            98.46,97.72,96.53,95.44,94.08,92.71,90.66,88.78,86.39,83.66,
            80.64,73.80,66.29,57.63,48.92,39.61,31.18,22.98,15.72,9.97,
            5.52,0.80,0.28,0.28
        ], dtype=float),
    },
    (5, 'conv9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            93.95,94.39,95.88,96.03,96.92,97.37,97.22,98.86,97.82,99.01,
            98.86,99.16,99.26,99.65,99.90,100.00,99.53,99.16,99.16,99.01,
            98.56,97.37,96.03,95.73,93.80,92.16,89.63,87.54,85.16,82.48,
            78.46,71.02,63.42,54.49,45.41,36.03,27.69,19.65,13.25,8.04,
            4.47,0.89,0.60,0.60
        ], dtype=float),
    },
    (5, 'flash6alt'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
            21,22,23,24,25,26,27,28,29,30,31,33,35,37,39,41,43,45,47,49,
            51,56,61,66
        ], dtype=float),
        'norm_pct': np.array([
            89.45,90.70,92.55,94.30,95.64,97.07,98.16,98.83,99.41,99.92,
            100.00,99.75,99.16,98.24,96.31,94.14,91.04,88.02,83.63,78.98,
            73.83,68.17,62.14,56.28,50.00,44.22,38.27,32.29,26.63,21.36,
            16.83,9.05,4.27,1.68,0.59,0.25,0.17,0.17,0.17,0.08,
            0.08,0.08,0.08,0.08
        ], dtype=float),
    },
    (2, 'flash9'): {
        'depth_mm': np.array([
            1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,17,19,21,23,25,27,29,31,35,39,43,47,51,61
        ], dtype=float),
        'norm_pct': np.array([
            96.81,97.33,97.96,98.48,98.80,99.32,99.63,100.00,99.79,99.69,
            99.32,98.59,97.80,96.49,94.82,89.69,82.88,75.34,67.07,59.37,
            51.88,44.45,37.80,25.89,16.10,8.19,2.98,0.68,0.13
        ], dtype=float),
    },
    (2, 'conv9'): {
    'depth_mm': np.array([
        5.6601,6.0139,6.3677,6.7214,7.0751,7.4289,7.7827,8.1364,8.4902,8.8439,
        9.1977,9.5514,9.9052,10.2590,10.6127,10.9665,11.3203,11.6740,12.0278,12.3815,
        12.7353,13.0890,13.4428,13.7965,14.1503,14.5041,14.8578,15.2116,15.5653,15.9191,
        16.2728,16.6266,16.9804,17.3341,17.6879,18.0416,18.3954,18.7491,19.1029,19.4566,
        19.8104,20.1642,20.5179,20.8717,21.2254,21.5792,21.9330,22.2867,22.6405,22.9942,
        23.3480,23.7017,24.0555,24.4093,24.7630,25.1168,25.4705,25.8243,26.1780,26.5318,
        26.8856,27.2393,27.5931,27.9468,28.3006,28.6543,29.0081,29.3619,29.7156,30.0694,
        30.4231,30.7769,31.1306,31.4844,31.8381,32.1919,32.5457,32.8994,33.2532,33.6069,
        33.9607,34.3145,34.6682,35.0220,35.3757,35.7295,36.0832,36.4370,36.7908,37.1445,
        37.4983,37.8520,38.2058,38.5595,38.9133,39.2671,39.6208,39.9746,40.3283,40.6821,
        41.0358,41.3896,41.7434,42.0971,42.4509,42.8046,43.1584,43.5121,43.8659,44.2196,
        44.5734,44.9272,45.2809,45.6347,45.9885,46.3422,46.6960,47.0497,47.4035,47.7572,
        48.1110,48.4647,48.8185,49.1723,49.5260,49.8798,50.2335,50.5873,50.9410,51.2948,
        51.6486,52.0023,52.3561,52.7098,53.0636,53.4173,53.7711,54.1249,54.4786,54.8324,
        55.1861,55.5399,55.8937,56.2474,56.6011,56.9549,57.3087,57.6624,58.0162,58.3699,
        58.7237,59.0775,59.4312,59.7850,60.1387
    ], dtype=float),

    'norm_pct': np.array([
        98.5652,98.6787,98.9405,99.0754,98.7389,99.4825,98.9048,99.3350,98.8820,98.9293,
        99.6467,100.0000,98.9994,98.7427,98.7750,99.0952,99.7004,99.1670,99.3333,99.6480,
        99.1558,98.4062,98.0374,98.0542,98.1861,98.0795,97.9480,97.5044,97.6282,97.7730,
        96.5196,97.0449,94.9409,94.3812,94.8283,94.2665,93.9342,93.1506,91.9909,91.4188,
        90.4036,88.9361,88.5987,87.8052,86.2484,85.1136,84.2875,83.8972,82.7994,81.5026,
        80.1138,78.9765,77.5314,76.5870,75.2636,74.3824,73.0624,71.6844,70.0944,68.4447,
        67.2361,66.2014,65.0456,63.6899,62.2384,61.2201,60.1821,58.7013,57.1457,56.1219,
        54.6703,53.6607,52.3535,50.9381,49.6950,48.5190,47.4848,46.0183,45.1706,43.8923,
        42.5658,41.5342,40.4781,39.3541,38.2628,37.0283,35.9589,35.0795,34.0191,33.0837,
        32.0362,31.0476,30.3006,29.5093,28.7081,28.2099,27.7366,26.8834,25.7104,24.8972,
        23.9704,23.1400,22.1669,21.3670,20.5253,19.6983,18.9410,18.1024,17.1572,16.2567,
        15.5621,14.7351,14.0684,13.2874,12.4961,11.8131,11.2182,10.6061,10.0177,9.4542,
        8.8107,8.1874,7.6695,7.1791,6.6134,6.1737,5.7928,5.4614,5.1722,4.7716,
        4.5231,4.2467,4.0168,3.8027,3.6437,3.3832,3.1489,3.1021,3.0419,2.8008,
        2.7079,2.6297,2.5356,2.5476,2.5270,2.4161,2.3202,2.2424,2.2467,2.2068,
        2.1840,2.2016,2.1689,2.1857,2.1582
    ], dtype=float),
    },
    (2, 'flash6alt'): {
        'depth_mm': np.array([
0.352944742,0.705889484,0.058834692,0.411779434,0.764724642,1.11766985,1.470614592,1.823559335,
2.176504077,2.529448819,2.882394027,3.235339235,3.588291427,3.941228719,4.294180912,4.647118203,
5.000071327,5.353008619,5.705960812,6.058898103,6.411850296,6.764788053,7.117740712,7.470678004,
7.823630196,8.176567954,8.529520146,8.882457438,9.235410096,9.588362289,9.941300046,10.29425224,
10.64718953,11.00014172,11.35307902,11.70603167,12.05896943,12.41192162,12.76485938,13.11781157,
13.47074887,13.82370106,14.17663882,14.52959147,14.88252877,15.23548096,15.58841825,15.94137091,
16.29430867,16.64726086,17.00019815,17.35315034,17.70608763,18.05904029,18.41197805,18.76493024,
19.11786753,19.47081973,19.82375748,20.17670968,20.52964743,20.88259963,21.23555182,21.58848958,
21.94144177,22.29437953,22.64733172,23.00026901,23.3532212,23.70615896,24.05911162,24.41204891,
24.7650011,25.1179384,25.47089105,25.82382881,26.176781,26.5297183,26.88267049,27.23560778,
27.58856044,27.9414982,28.29445039,28.64738768,29.00033987,29.35327763,29.70622982,30.05916758,
30.41211977,30.76505753,31.11800972,31.47094702,31.82389967,32.17683697,32.52978962,32.88274182,
33.23567911,33.5886313,33.94156906,34.29452125,34.64745901,35.0004112,35.35334896,35.70630115,
36.05923844,36.41219064,36.76512839,37.11808105,37.47101834,37.82397054,38.17690783,38.52986049,
38.88279778,39.23575044,39.58868773,39.94163992,40.29457721,40.64752987,41.00046716,41.35341982,
41.70635711,42.05930977,42.41224706,42.76519925,43.11813701,43.4710892,43.82402696,44.17697915,
44.52993135,44.88286864,45.2358213,45.58875859,45.94171125,46.29464854,46.6476012,47.00053849,
47.35349068,47.70642797,48.05938063,48.41231839,48.76527058,49.11820787,49.47116007,49.82409782,
50.17705002,50.52998777,50.88293997,51.23587726,51.58882945,51.94176721,52.2947194,52.64765716,
53.00060935,53.35354711,53.7064993,54.05943659,54.41238925,54.76532654,55.1182792,55.47121649,
55.82416869,56.17712088,56.53005864,56.88301083,57.23594859,57.58890078,57.94183854
], dtype=float),

'norm_pct': np.array([
94.40331203,95.45158567,95.92990072,95.23614293,95.62763777,96.05369991,96.4492142,96.35113952,
96.29808272,97.02319225,97.84878813,98.13577716,97.99590016,98.43000121,98.18401061,97.86406206,
97.81422083,98.74110696,99.05140882,99.85449576,99.62458298,99.17922746,100,99.86414245,
99.94855099,99.9565899,99.43084529,98.99995981,98.61650388,98.11326822,97.66871659,97.61887536,
96.98540938,96.34551228,95.42666506,95.08742313,94.02628723,93.011777,91.37344749,90.55267495,
89.64347442,88.68845211,87.50673259,85.57337514,83.85385265,82.35218457,81.97515977,79.60127015,
77.95570562,76.4226858,74.42260541,72.80356928,70.7319426,69.15953214,68.15788416,66.49463403,
65.11917682,63.1649182,60.83765425,59.15350295,57.85441537,56.53040717,54.99095623,53.12432172,
50.76972547,48.66996262,47.42473572,45.84509024,43.6215282,42.10137063,40.59407532,38.54013425,
36.86161019,35.32296314,33.30198159,31.42007315,29.78093975,28.24791993,26.75348688,25.11354958,
23.81928534,22.57486233,20.96949234,19.68487479,18.2378713,16.88090357,15.9138229,14.58579525,
13.52224768,12.70790627,11.46991439,10.42244463,9.797821456,9.282527433,8.329916797,7.632139555,
7.214116323,6.698018409,6.145745408,5.667430363,5.563728446,5.419831987,5.205193135,4.861127859,
4.32091322,4.188271233,4.448731862,4.520278146,4.336991037,4.193898469,4.242131918,4.464005788,
4.152096145,4.133606656,4.150488364,4.196310141,4.150488364,4.198721814,4.054021464,3.984082962,
4.138430001,3.984886852,3.988102416,3.996945215,3.921379477,3.969612927,4.054021464,3.832951485,
3.770248,3.781502472,3.689055026,3.805619197,3.84742152,3.865911009,3.763012983,3.670565537,
3.786325817,4.331363801,3.835363158,3.813658105,3.845813738,4.267856425,4.154507818,3.668957756,
3.71156397,4.058844809,3.827324249,3.768640219,3.706740625,3.838578721,3.771855782,4.01141525,
4.104666586,4.051609791,3.993729652,3.904497769,3.835363158,3.914144459,3.707544515,3.775875236,
3.718798987,3.963985691,3.906909442,3.93424173,4.04035532,3.713975642,3.904497769
], dtype=float),
    },
}

sorted(MEASURED_PDD.keys())



## 3. Shared file-loading and profile helper functions


In [ ]:
def _safe_read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

    df = pd.read_csv(
        path,
        sep=r'[\s,;]+',
        engine='python',
        comment='#',
        header=None,
    )
    df = df.dropna(axis=1, how='all')
    if df.empty:
        raise ValueError(f'No numeric rows found in {path}')
    return df


def _looks_like_mesh_scorer(df):
    return df.shape[1] >= 4 and df.shape[1] <= 6


def _pick_profile_axis(df):
    axes = {0: df.iloc[:, 0].nunique(), 1: df.iloc[:, 1].nunique(), 2: df.iloc[:, 2].nunique()}
    best_axis = max(axes, key=axes.get)
    return best_axis


def read_pdd_file(path, depth_bin_mm=1.0, depth_offset_mm=0.0):
    df = _safe_read_table(path)

    if df.shape[1] == 2:
        out = pd.DataFrame({'depth_mm': df.iloc[:, 0].astype(float), 'dose': df.iloc[:, 1].astype(float)})
        return out.sort_values('depth_mm').drop_duplicates('depth_mm').reset_index(drop=True)

    if _looks_like_mesh_scorer(df):
        iZ = df.iloc[:, 2].astype(float)
        dose = df.iloc[:, 3].astype(float)
        out = pd.DataFrame({'depth_idx': iZ, 'dose': dose})
        out = out.groupby('depth_idx', as_index=False)['dose'].mean()
        out['depth_mm'] = out['depth_idx'] * depth_bin_mm + depth_offset_mm
        out = out[['depth_mm', 'dose']].sort_values('depth_mm').reset_index(drop=True)
        return out

    raise ValueError('Unrecognized PDD format. Expected 2 columns or mesh scorer columns (iX,iY,iZ,value,...).')


def read_profile_file(path, y_name='value', position_axis=None, position_bin_mm=1.0, position_offset_mm=0.0):
    df = _safe_read_table(path)

    if df.shape[1] == 2:
        out = pd.DataFrame({'position_mm': df.iloc[:, 0].astype(float), y_name: df.iloc[:, 1].astype(float)})
        return out.sort_values('position_mm').drop_duplicates('position_mm').reset_index(drop=True)

    if _looks_like_mesh_scorer(df):
        axis_map = {'ix': 0, 'iy': 1, 'iz': 2}
        if position_axis is None:
            axis_col = _pick_profile_axis(df)
        else:
            axis_col = axis_map[str(position_axis).lower()]

        pos_idx = df.iloc[:, axis_col].astype(float)
        val = df.iloc[:, 3].astype(float)
        out = pd.DataFrame({'pos_idx': pos_idx, y_name: val})

        if df.shape[1] >= 6:
            sum_val = df.iloc[:, 3].astype(float)
            sum_sq = df.iloc[:, 4].astype(float)
            entries = df.iloc[:, 5].astype(float)

            mean_val = sum_val / entries.replace(0, np.nan)
            variance = (sum_sq / entries.replace(0, np.nan)) - mean_val ** 2
            sigma_mean = np.sqrt(np.clip(variance, 0.0, None) / entries.replace(0, np.nan))
            rel_unc_pct = 100.0 * sigma_mean / mean_val.replace(0, np.nan).abs()
            out['mc_rel_uncertainty_pct'] = rel_unc_pct.replace([np.inf, -np.inf], np.nan)

        agg = {y_name: 'mean'}
        if 'mc_rel_uncertainty_pct' in out:
            agg['mc_rel_uncertainty_pct'] = 'mean'

        out = out.groupby('pos_idx', as_index=False).agg(agg)
        out['position_mm'] = out['pos_idx'] * position_bin_mm + position_offset_mm
        cols = ['position_mm', y_name]
        if 'mc_rel_uncertainty_pct' in out:
            cols.append('mc_rel_uncertainty_pct')
        out = out[cols].sort_values('position_mm').reset_index(drop=True)
        return out

    raise ValueError(f'Unrecognized profile format in {path}. Expected 2 columns or mesh scorer columns.')



PROFILE_FILE_PREFIX_BY_DEPTH = {
    "dmax": "latDmax",
    "r50": "latR50",
    "middepth": "latMid",
}


def _extract_history_count(filename):
    match = re.search(r"_N(\d+(?:\.\d+)?)([kKmMbB]?)", str(filename))
    if not match:
        return -1.0

    value = float(match.group(1))
    suffix = match.group(2).lower()
    scale = {"": 1.0, "k": 1e3, "m": 1e6, "b": 1e9}[suffix]
    return value * scale


def discover_lateral_profile_files(
    profile_mode,
    applicator_cm,
    search_dir=".",
    depth_keys=("dmax", "r50", "middepth"),
):
    """Find lateral simulation profile files from names like latDmax_flash9_10cm_..._N500M.txt."""
    search_dir = Path(search_dir)
    mode_text = str(profile_mode).lower()
    applicator_text = f"{applicator_cm:g}cm".lower()

    files = list(search_dir.glob("lat*.txt")) + list(search_dir.glob("build/lat*.txt"))
    selected = {}

    for depth_key in depth_keys:
        prefix = PROFILE_FILE_PREFIX_BY_DEPTH[depth_key].lower()
        matches = []

        for path in files:
            name = path.name.lower()
            if not name.startswith(prefix.lower()):
                continue
            if mode_text not in name:
                continue
            if applicator_text not in name:
                continue
            matches.append(path)

        if not matches:
            raise FileNotFoundError(
                f"No {depth_key} lateral file found for mode={profile_mode!r}, "
                f"applicator={applicator_text!r} in {search_dir.resolve()}"
            )

        matches = sorted(
            matches,
            key=lambda path: (_extract_history_count(path.name), path.name),
            reverse=True,
        )
        selected[depth_key] = str(matches[0])

        if len(matches) > 1:
            print(f"Auto-selected {depth_key}: {matches[0]} from {len(matches)} matches")
        else:
            print(f"Auto-selected {depth_key}: {matches[0]}")

    return selected


def measured_lateral_columns_for_applicator(applicator_cm):
    diameter_text = f"{applicator_cm:g}cm"
    return {
        "dmax": {"pos": f"pos_{diameter_text}_dmax", "dose": f"dose_{diameter_text}_dmax"},
        "r50": {"pos": f"pos_{diameter_text}_r50", "dose": f"dose_{diameter_text}_r50"},
        "middepth": {"pos": f"pos_{diameter_text}_middepth", "dose": f"dose_{diameter_text}_middepth"},
    }


## 4. Analysis settings

Edit this cell first when changing applicator size, measured Excel input, depth columns, or simulation profile files.


In [ ]:
# ============================================================
# PUBLICATION-SAFE LATERAL PROFILE SETTINGS
# Copy-paste this cell BEFORE the main LOAD / ANALYZE loop.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# 1) Choose which measured sheet/config you are analyzing
# ------------------------------------------------------------
lateral_profile_depth_keys = ('dmax', 'r50', 'middepth')

lateral_mode_select = 'flash'
lateral_applicator_select_cm = 10

measured_lateral_excel_file = "simsim2.xlsx"
measured_lateral_excel_sheet = "flash9"   # change to "flash6" for 6 MeV data

# Automatically use the measured Excel columns for the selected applicator.
# Example: lateral_applicator_select_cm = 10 uses pos_10cm_dmax/dose_10cm_dmax, etc.
measured_lateral_excel_columns = measured_lateral_columns_for_applicator(
    lateral_applicator_select_cm
)

# ------------------------------------------------------------
# 2) Simulated files
# ------------------------------------------------------------
# Easy mode: choose the measured sheet/profile mode and applicator above,
# then let the notebook find matching files named like:
#   latDmax_flash9_10cm_..._N500M_....txt
#   latR50_flash9_10cm_..._N500M_....txt
#   latMid_flash9_10cm_..._N500M_....txt
#
# If multiple files match, the notebook chooses the one with the largest N count
# in the filename, e.g. N500M beats N100M.
# ------------------------------------------------------------
auto_discover_sim_lateral_files = True
sim_lateral_search_dir = Path(".")

if auto_discover_sim_lateral_files:
    sim_lateral_files = discover_lateral_profile_files(
        profile_mode=measured_lateral_excel_sheet,
        applicator_cm=lateral_applicator_select_cm,
        search_dir=sim_lateral_search_dir,
        depth_keys=lateral_profile_depth_keys,
    )
else:
    # Manual fallback if you ever want to override auto-discovery.
    sim_lateral_files = {
        'dmax': 'latDmax_flash9_10cm_r3001_E9_70_sE1_30_sTh0_2_10cm_N500M_J12766259_T1.txt',
        'r50': 'latR50_flash9_10cm_r3001_E9_70_sE1_30_sTh0_2_10cm_N500M_J12766259_T1.txt',
        'middepth': 'latMid_flash9_10cm_r3001_E9_70_sE1_30_sTh0_2_10cm_N500M_J12766259_T1.txt',
    }

measured_lateral_files = {
    'dmax': None,
    'r50': None,
    'middepth': None,
}

# ------------------------------------------------------------
# 3) Do NOT alter measured data
# ------------------------------------------------------------
apply_measured_noise = False

# Use 'none' first for debugging.
# Only use 'subtract_min' if you can justify a measured detector baseline offset.
measured_baseline_mode = 'none'

measured_y_shift_pct = {
    'dmax': 0.0,
    'r50': 0.0,
    'middepth': 0.0,
}

clip_profile_y_to_0_100 = True

# ------------------------------------------------------------
# 4) Coordinate alignment
# ------------------------------------------------------------
# Center both profiles using the half-maximum midpoint.
auto_center_profiles = True

# Keep fitting physically reasonable:
# no width stretching, only a small left/right shift.
auto_fit_sim_to_measured = True
sim_fit_scale_grid = np.array([1.0])
sim_fit_shift_grid_mm = np.linspace(-5.0, 5.0, 101)

# Manual x corrections after centering, if needed.
# Keep these at 1 and 0 unless you have a documented reason.
sim_manual_x_scale = {'dmax': 1.0, 'r50': 1.0, 'middepth': 1.0}
sim_manual_x_shift_mm = {'dmax': 0.0, 'r50': 0.0, 'middepth': 0.0}

meas_manual_x_scale = {'dmax': 1.0, 'r50': 1.0, 'middepth': 1.0}
meas_manual_x_shift_mm = {'dmax': 0.0, 'r50': 0.0, 'middepth': 0.0}

# ------------------------------------------------------------
# 5) Scorer/profile axis calibration
# ------------------------------------------------------------
# Your lateral mesh is:
# /score/mesh/boxSize 75 2.5 0.5 mm
# /score/mesh/nBin    150 1 1
#
# If your read_profile_file uses bin index directly, 1 mm/bin is correct
# for a 150 mm full width / 150 bins.
profile_axis = None
profile_position_bin_mm = 1.0
profile_position_offset_mm = 0.0

sim_position_scale = 1.0
sim_position_shift_mm = 0.0

# ------------------------------------------------------------
# 6) Cropping/trimming
# ------------------------------------------------------------
# Keep full profile for now.
meas_crop_min_mm = {'dmax': None, 'r50': None, 'middepth': None}
meas_crop_max_mm = {'dmax': None, 'r50': None, 'middepth': None}

sim_crop_min_mm = {'dmax': None, 'r50': None, 'middepth': None}
sim_crop_max_mm = {'dmax': None, 'r50': None, 'middepth': None}

trim_meas_by_threshold = {'dmax': False, 'r50': False, 'middepth': False}
trim_sim_by_threshold  = {'dmax': False, 'r50': False, 'middepth': False}

meas_trim_threshold_pct = {'dmax': 1.0, 'r50': 1.0, 'middepth': 1.0}
sim_trim_threshold_pct  = {'dmax': 1.0, 'r50': 1.0, 'middepth': 1.0}

# ------------------------------------------------------------
# 7) Gamma / smoothing
# ------------------------------------------------------------
gamma_dose_percent = 2.0
gamma_dist_mm = 2.0

profile_smooth_window_mm = 0.0
resid_smooth_window_mm = 0.0     # keep raw first
gamma_smooth_window_mm = 0.0     # keep raw first

# Keep full profile gamma. If later you want to exclude very low dose tails,
# set e.g. analysis_dose_threshold_pct = 5.0.
analysis_dose_threshold_pct = None

# ------------------------------------------------------------
# 8) Plot limits/appearance
# ------------------------------------------------------------
profile_xlim = (-180, 180)
profile_ylim = (0, 105)
resid_ylim = (-20, 20)
gamma_ylim = (0, 2.0)

color_meas = '#0072B2'
color_sim = '#D55E00'
color_res = '#0072B2'
color_gamma = '#0072B2'

lw_meas = 4
lw_sim = 3.6
lw_secondary = 2
lw_ref = 2

ls_meas = '-'
ls_sim = '--'

show_meas_markers = False
meas_marker = 'o'
meas_marker_size = 2.8

title_fs = 12
label_fs = 11
tick_fs = 10
legend_fs = 9

# ------------------------------------------------------------
# 9) CRITICAL: disable measured-tail replacement
# ------------------------------------------------------------
# Your old function made measured tails follow the simulation.
# That is not valid for real validation.
# This override keeps measured data unchanged even if the old analysis loop
# still calls make_measured_tails_follow_sim(...).
def make_measured_tails_follow_sim(meas_x, meas_y, sim_x, sim_y, tail_start_pct=98.0):
    """
    Publication-safe override:
    return measured profile unchanged.
    Do not graft simulation tails onto measured data.
    """
    return np.asarray(meas_x, dtype=float).copy(), np.asarray(meas_y, dtype=float).copy()

print("Publication-safe lateral settings loaded.")
print("Measured sheet:", measured_lateral_excel_sheet)
print("Applicator:", lateral_applicator_select_cm, "cm")
print("Measured columns:")
for k, v in measured_lateral_excel_columns.items():
    print(f"  {k}: {v['pos']} / {v['dose']}")

## 5. Optional measured-profile diagnostic

Use this section to confirm that measured 10 cm, 5 cm, and 2 cm profiles are distinct before processing.


In [ ]:
# ============================================================
# QUICK CHECK:
# Are measured 10 cm, 5 cm, and 2 cm profiles the same or different?
# ------------------------------------------------------------
# This checks dmax, R50, and mid-depth directly from the Excel sheet.
#
# It compares:
#   - raw column values
#   - normalized profiles
#   - interpolated profiles on a common x-grid
#
# Use this to make sure you are not accidentally plotting the same
# measured profile for 10 cm, 5 cm, and 2 cm.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# Excel file/sheet
# ------------------------------------------------------------
excel_file = measured_lateral_excel_file
excel_sheet = measured_lateral_excel_sheet

xdf_check = pd.read_excel(excel_file, sheet_name=excel_sheet)

print("Excel file:", excel_file)
print("Excel sheet:", excel_sheet)
print("\nAvailable columns:")
print(list(xdf_check.columns))

# ------------------------------------------------------------
# Define measured columns for each applicator
# ------------------------------------------------------------
# Edit these names if your Excel columns are named differently.
# ------------------------------------------------------------

applicator_columns = {
    "10cm": {
        "dmax": {
            "pos": "pos_10cm_dmax",
            "dose": "dose_10cm_dmax",
        },
        "r50": {
            "pos": "pos_10cm_r50",
            "dose": "dose_10cm_r50",
        },
        "middepth": {
            "pos": "pos_10cm_middepth",
            "dose": "dose_10cm_middepth",
        },
    },
    "5cm": {
        "dmax": {
            "pos": "pos_5cm_dmax",
            "dose": "dose_5cm_dmax",
        },
        "r50": {
            "pos": "pos_5cm_r50",
            "dose": "dose_5cm_r50",
        },
        "middepth": {
            "pos": "pos_5cm_middepth",
            "dose": "dose_5cm_middepth",
        },
    },
    "2cm": {
        "dmax": {
            "pos": "pos_2cm_dmax",
            "dose": "dose_2cm_dmax",
        },
        "r50": {
            "pos": "pos_2cm_r50",
            "dose": "dose_2cm_r50",
        },
        "middepth": {
            "pos": "pos_2cm_middepth",
            "dose": "dose_2cm_middepth",
        },
    },
}

depth_keys_to_check = ["dmax", "r50", "middepth"]
app_keys_to_check = ["10cm", "5cm", "2cm"]

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def normalize_to_100_check(y):
    y = np.asarray(y, dtype=float)

    ymax = np.nanmax(y)

    if not np.isfinite(ymax) or ymax == 0:
        return np.zeros_like(y)

    return 100.0 * y / ymax


def read_profile_from_excel_check(df, pos_col, dose_col):
    if pos_col not in df.columns:
        raise KeyError(f"Missing position column: {pos_col}")

    if dose_col not in df.columns:
        raise KeyError(f"Missing dose column: {dose_col}")

    out = df[[pos_col, dose_col]].dropna().copy()
    out.columns = ["x", "y"]
    out = out.sort_values("x").drop_duplicates("x").reset_index(drop=True)

    return out["x"].values.astype(float), out["y"].values.astype(float)


def center_by_peak_check(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) == 0:
        return x

    i_peak = int(np.nanargmax(y))
    center = x[i_peak]

    return x - center


def compare_two_profiles(label_a, x_a, y_a, label_b, x_b, y_b):
    """
    Compare two profiles both exactly and after normalization/interpolation.
    """
    x_a = np.asarray(x_a, dtype=float)
    y_a = np.asarray(y_a, dtype=float)
    x_b = np.asarray(x_b, dtype=float)
    y_b = np.asarray(y_b, dtype=float)

    # Exact same length and values?
    exact_same = (
        len(x_a) == len(x_b)
        and len(y_a) == len(y_b)
        and np.allclose(x_a, x_b, rtol=0, atol=0, equal_nan=True)
        and np.allclose(y_a, y_b, rtol=0, atol=0, equal_nan=True)
    )

    # Normalize to 100 and center by peak for shape comparison
    y_a_norm = normalize_to_100_check(y_a)
    y_b_norm = normalize_to_100_check(y_b)

    x_a_ctr = center_by_peak_check(x_a, y_a_norm)
    x_b_ctr = center_by_peak_check(x_b, y_b_norm)

    # Common overlap region
    xmin = max(np.nanmin(x_a_ctr), np.nanmin(x_b_ctr))
    xmax = min(np.nanmax(x_a_ctr), np.nanmax(x_b_ctr))

    if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax <= xmin:
        return {
            "pair": f"{label_a} vs {label_b}",
            "exact_same": exact_same,
            "common_points": 0,
            "shape_rmse_pct": np.nan,
            "shape_maxabs_pct": np.nan,
            "nearly_same_shape": False,
        }

    # Common grid using the coarser median spacing
    dx_a = np.nanmedian(np.diff(np.sort(x_a_ctr)))
    dx_b = np.nanmedian(np.diff(np.sort(x_b_ctr)))

    dx = np.nanmax([abs(dx_a), abs(dx_b)])

    if not np.isfinite(dx) or dx <= 0:
        dx = 1.0

    x_common = np.arange(xmin, xmax + 0.5 * dx, dx)

    ya_common = np.interp(x_common, x_a_ctr, y_a_norm)
    yb_common = np.interp(x_common, x_b_ctr, y_b_norm)

    diff = ya_common - yb_common

    rmse = float(np.sqrt(np.nanmean(diff ** 2)))
    maxabs = float(np.nanmax(np.abs(diff)))

    nearly_same_shape = bool(rmse < 0.25 and maxabs < 1.0)

    return {
        "pair": f"{label_a} vs {label_b}",
        "exact_same": exact_same,
        "common_points": len(x_common),
        "shape_rmse_pct": rmse,
        "shape_maxabs_pct": maxabs,
        "nearly_same_shape": nearly_same_shape,
    }


# ------------------------------------------------------------
# Read all profiles
# ------------------------------------------------------------

profiles_by_app = {}
summary_rows = []

for app in app_keys_to_check:
    profiles_by_app[app] = {}

    for depth in depth_keys_to_check:
        cols = applicator_columns[app][depth]
        pos_col = cols["pos"]
        dose_col = cols["dose"]

        try:
            x, y = read_profile_from_excel_check(
                xdf_check,
                pos_col,
                dose_col,
            )

            profiles_by_app[app][depth] = {
                "x": x,
                "y": y,
                "pos_col": pos_col,
                "dose_col": dose_col,
            }

            summary_rows.append({
                "applicator": app,
                "depth": depth,
                "pos_col": pos_col,
                "dose_col": dose_col,
                "n_points": len(x),
                "x_min": float(np.nanmin(x)),
                "x_max": float(np.nanmax(x)),
                "dose_min": float(np.nanmin(y)),
                "dose_max": float(np.nanmax(y)),
            })

        except Exception as exc:
            print(f"[WARNING] Could not read {app} {depth}: {exc}")

            profiles_by_app[app][depth] = None

            summary_rows.append({
                "applicator": app,
                "depth": depth,
                "pos_col": pos_col,
                "dose_col": dose_col,
                "n_points": np.nan,
                "x_min": np.nan,
                "x_max": np.nan,
                "dose_min": np.nan,
                "dose_max": np.nan,
            })

summary_df = pd.DataFrame(summary_rows)

print("\nProfile summary:")
display(summary_df.round(4))

# ------------------------------------------------------------
# Pairwise comparisons
# ------------------------------------------------------------

comparison_rows = []

for depth in depth_keys_to_check:
    pairs = [
        ("10cm", "5cm"),
        ("10cm", "2cm"),
        ("5cm", "2cm"),
    ]

    for app_a, app_b in pairs:
        pa = profiles_by_app[app_a].get(depth)
        pb = profiles_by_app[app_b].get(depth)

        if pa is None or pb is None:
            comparison_rows.append({
                "depth": depth,
                "pair": f"{app_a} vs {app_b}",
                "exact_same": np.nan,
                "common_points": np.nan,
                "shape_rmse_pct": np.nan,
                "shape_maxabs_pct": np.nan,
                "nearly_same_shape": np.nan,
            })
            continue

        comp = compare_two_profiles(
            app_a,
            pa["x"],
            pa["y"],
            app_b,
            pb["x"],
            pb["y"],
        )

        comp["depth"] = depth
        comparison_rows.append(comp)

comparison_df = pd.DataFrame(comparison_rows)

comparison_df = comparison_df[
    [
        "depth",
        "pair",
        "exact_same",
        "nearly_same_shape",
        "common_points",
        "shape_rmse_pct",
        "shape_maxabs_pct",
    ]
]

print("\nPairwise profile comparison:")
display(comparison_df.round(4))

# ------------------------------------------------------------
# Plot overlays for quick visual check
# ------------------------------------------------------------

colors = {
    "10cm": "#0072B2",
    "5cm": "#E69F00",
    "2cm": "#009E73",
}

fig, axes = plt.subplots(
    1,
    len(depth_keys_to_check),
    figsize=(16, 4.2),
    sharey=True,
)

if len(depth_keys_to_check) == 1:
    axes = [axes]

for ax, depth in zip(axes, depth_keys_to_check):
    for app in app_keys_to_check:
        p = profiles_by_app[app].get(depth)

        if p is None:
            continue

        x = p["x"]
        y = normalize_to_100_check(p["y"])
        x_ctr = center_by_peak_check(x, y)

        ax.plot(
            x_ctr,
            y,
            lw=2.5,
            color=colors[app],
            label=app,
        )

    ax.axvline(0, color="0.6", ls="--", lw=1.0)
    ax.set_title(depth)
    ax.set_xlabel("Centered position by peak (mm)")
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.25)

axes[0].set_ylabel("Normalized dose (%)")
axes[-1].legend(frameon=False, loc="best")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Final interpretation helper
# ------------------------------------------------------------

print("\nInterpretation:")
for _, row in comparison_df.iterrows():
    depth = row["depth"]
    pair = row["pair"]

    if row["exact_same"] is True:
        print(f"  [ALERT] {depth}: {pair} are EXACTLY identical.")
    elif row["nearly_same_shape"] is True:
        print(
            f"  [CHECK] {depth}: {pair} are not exactly identical, "
            f"but their normalized shapes are nearly the same "
            f"(RMSE={row['shape_rmse_pct']:.3f}%, MaxAbs={row['shape_maxabs_pct']:.3f}%)."
        )
    else:
        print(
            f"  [OK] {depth}: {pair} appear different "
            f"(RMSE={row['shape_rmse_pct']:.3f}%, MaxAbs={row['shape_maxabs_pct']:.3f}%)."
        )

## 6. Reset stale processed results


In [ ]:
# ============================================================
# RESET OLD PROCESSED PROFILE RESULTS
# ------------------------------------------------------------
# Run this after changing applicator / measured columns / sim files.
#
# This prevents old 10 cm processed dictionaries from being reused
# when you are trying to plot 2 cm or 5 cm.
# ============================================================

old_result_names = [
    "results",
    "edge_results",
    "final_results",
    "exact_plot_results",
    "manual_tuned_results",
    "tail_tuned_results",
    "flipped_tail_tuned_results",
    "profile_only_results",
    "final_clean_results",
    "profile_diff_gamma_results",
    "dmax_right_tail_results",
]

for name in old_result_names:
    if name in globals():
        del globals()[name]
        print(f"Deleted old {name}")

print("Old processed result dictionaries cleared.")
print("Now rerun the cell that creates results from measured Excel + simulation files.")

## 7. Build fresh measured and simulation results


In [ ]:
# ============================================================
# CREATE FRESH results FROM CURRENT MEASURED EXCEL + SIM FILES
# ------------------------------------------------------------
# Run this after:
#   1) settings cell
#   2) diagnostic measured Excel cell if you want
#   3) reset old dictionaries cell
#
# This creates a fresh "results" dictionary for the current
# applicator and simulation files.
# ============================================================

import numpy as np
import pandas as pd
import inspect
from pathlib import Path

# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------
required_names = [
    "lateral_profile_depth_keys",
    "measured_lateral_excel_file",
    "measured_lateral_excel_sheet",
    "measured_lateral_excel_columns",
    "sim_lateral_files",
]

for name in required_names:
    if name not in globals():
        raise RuntimeError(f"Missing required setting: {name}")

if "edge_level_pct" not in globals():
    edge_level_pct = 50.0

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def normalize_to_100(y):
    y = np.asarray(y, dtype=float)

    ymax = np.nanmax(y)

    if not np.isfinite(ymax) or ymax == 0:
        return np.zeros_like(y)

    return 100.0 * y / ymax


def get_left_right_crossings_local(x, y, level_pct=50.0):
    x = np.asarray(x, dtype=float)
    y = normalize_to_100(y)

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    if len(x) < 4 or np.nanmax(y) <= 0:
        return np.nan, np.nan

    i_peak = int(np.nanargmax(y))
    level = float(level_pct)

    x_left = np.nan
    x_right = np.nan

    # Left crossing
    for i in range(i_peak, 0, -1):
        y1 = y[i - 1]
        y2 = y[i]

        if (y1 <= level <= y2) or (y2 <= level <= y1):
            x1 = x[i - 1]
            x2 = x[i]

            if y2 == y1:
                x_left = x1
            else:
                t = (level - y1) / (y2 - y1)
                x_left = x1 + t * (x2 - x1)

            break

    # Right crossing
    for i in range(i_peak, len(y) - 1):
        y1 = y[i]
        y2 = y[i + 1]

        if (y1 >= level >= y2) or (y2 >= level >= y1):
            x1 = x[i]
            x2 = x[i + 1]

            if y2 == y1:
                x_right = x1
            else:
                t = (level - y1) / (y2 - y1)
                x_right = x1 + t * (x2 - x1)

            break

    return float(x_left), float(x_right)


def center_profile_by_edges_local(x, y, level_pct=50.0):
    left, right = get_left_right_crossings_local(
        x,
        y,
        level_pct=level_pct,
    )

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if np.isfinite(left) and np.isfinite(right):
        center = 0.5 * (left + right)
    else:
        center = float(x[int(np.nanargmax(y))])

    return x - center, center


def read_profile_compatible_local(path, y_name="value"):
    if "read_profile_file" not in globals():
        raise RuntimeError(
            "read_profile_file() is not defined. "
            "Run the helper cell that defines read_profile_file first."
        )

    sig = inspect.signature(read_profile_file)
    params = sig.parameters

    kwargs = {"y_name": y_name}

    if "position_axis" in params:
        kwargs["position_axis"] = globals().get("profile_axis", None)

    if "position_bin_mm" in params:
        kwargs["position_bin_mm"] = globals().get("profile_position_bin_mm", 1.0)

    if "position_offset_mm" in params:
        kwargs["position_offset_mm"] = globals().get("profile_position_offset_mm", 0.0)

    return read_profile_file(path, **kwargs).copy()


def read_measured_from_excel_local(key):
    cols = measured_lateral_excel_columns[key]
    pos_col = cols["pos"]
    dose_col = cols["dose"]

    if "xdf" in globals():
        df = xdf.copy()
    else:
        df = pd.read_excel(
            measured_lateral_excel_file,
            sheet_name=measured_lateral_excel_sheet,
        )

    if pos_col not in df.columns:
        raise KeyError(f"[{key}] Missing measured position column: {pos_col}")

    if dose_col not in df.columns:
        raise KeyError(f"[{key}] Missing measured dose column: {dose_col}")

    out = df[[pos_col, dose_col]].dropna().copy()
    out.columns = ["position_mm", "meas"]

    out = (
        out
        .sort_values("position_mm")
        .drop_duplicates("position_mm")
        .reset_index(drop=True)
    )

    return out


# ------------------------------------------------------------
# Create fresh results
# ------------------------------------------------------------

results = {}

print("Creating fresh results using:")
print("  measured_lateral_excel_file:", measured_lateral_excel_file)
print("  measured_lateral_excel_sheet:", measured_lateral_excel_sheet)
print("  lateral_applicator_select_cm:", lateral_applicator_select_cm)
print("  sim_lateral_files:")

for k, v in sim_lateral_files.items():
    print(f"    {k}: {v}")

for key in lateral_profile_depth_keys:
    # --------------------------
    # Measured
    # --------------------------
    mdf = read_measured_from_excel_local(key)

    meas_x_raw = mdf["position_mm"].values
    meas_y_raw = mdf["meas"].values

    meas_y = normalize_to_100(meas_y_raw)

    meas_x, meas_center = center_profile_by_edges_local(
        meas_x_raw,
        meas_y,
        level_pct=edge_level_pct,
    )

    # --------------------------
    # Simulation
    # --------------------------
    if key not in sim_lateral_files:
        raise KeyError(f"Missing simulation file for key: {key}")

    sim_path = Path(sim_lateral_files[key])

    if not sim_path.exists():
        print(f"WARNING: simulation file does not exist: {sim_path}")

    sdf = read_profile_compatible_local(
        sim_lateral_files[key],
        y_name="sim",
    )

    sim_position_scale_local = globals().get("sim_position_scale", 1.0)
    sim_position_shift_local = globals().get("sim_position_shift_mm", 0.0)

    sim_x_raw = (
        sdf["position_mm"].values * sim_position_scale_local
        + sim_position_shift_local
    )

    sim_y_raw = sdf["sim"].values

    sim_y = normalize_to_100(sim_y_raw)

    sim_x, sim_center = center_profile_by_edges_local(
        sim_x_raw,
        sim_y,
        level_pct=edge_level_pct,
    )

    results[key] = {
        "meas_x": meas_x,
        "meas_y": meas_y,
        "sim_x": sim_x,
        "sim_y": sim_y,
        "meas_center_mm": meas_center,
        "sim_center_mm": sim_center,
    }

print("\nCreated fresh results successfully.")
print("results keys:", list(results.keys()))

for key in results:
    print(
        f"{key}: "
        f"meas points={len(results[key]['meas_x'])}, "
        f"sim points={len(results[key]['sim_x'])}, "
        f"meas x range=({np.nanmin(results[key]['meas_x']):.2f}, {np.nanmax(results[key]['meas_x']):.2f}), "
        f"sim x range=({np.nanmin(results[key]['sim_x']):.2f}, {np.nanmax(results[key]['sim_x']):.2f})"
    )

## 8. Optional raw measured Excel preview


In [ ]:
# ============================================================
# DIAGNOSTIC: plot raw measured Excel profiles before processing
# ============================================================

xdf = pd.read_excel(measured_lateral_excel_file, sheet_name=measured_lateral_excel_sheet)

print("Available Excel columns:")
print(list(xdf.columns))

for key in lateral_profile_depth_keys:
    cols = measured_lateral_excel_columns[key]
    pos_col = cols['pos']
    dose_col = cols['dose']

    if pos_col not in xdf.columns or dose_col not in xdf.columns:
        print(f"[{key}] Missing columns: {pos_col}, {dose_col}")
        continue

    raw = xdf[[pos_col, dose_col]].dropna().copy()
    raw = raw.sort_values(pos_col)

    plt.figure(figsize=(5, 3))
    plt.plot(raw[pos_col], raw[dose_col], 'o-', ms=3)
    plt.title(f"RAW measured {key}: {pos_col}, {dose_col}")
    plt.xlabel("Raw measured position (mm)")
    plt.ylabel("Raw measured dose")
    plt.grid(True, alpha=0.3)
    plt.show()

    print(f"\n{key} raw measured summary:")
    display(raw.describe())

## 9. Select processed result source


In [ ]:
# ============================================================
# FIX: choose available profile results source
# ------------------------------------------------------------
# The edge-alignment cell expects a dictionary called "results".
# If you do not have results, this will reuse whichever processed
# result dictionary exists in your notebook.
# ============================================================

if "results" in globals():
    print("Using existing results")
    
elif "final_results" in globals():
    results = final_results
    print("Created results = final_results")
    
elif "edge_results" in globals():
    results = edge_results
    print("Created results = edge_results")
    
elif "exact_plot_results" in globals():
    results = exact_plot_results
    print("Created results = exact_plot_results")
    
elif "manual_tuned_results" in globals():
    # Convert manual_tuned_results into the structure expected by your edge cell
    results = {}
    
    for key, r in manual_tuned_results.items():
        results[key] = {
            "meas_x": r["meas_x"],
            "meas_y": r["meas_y"],
            "sim_x": r["sim_x_tuned"],
            "sim_y": r["sim_y_tuned"],
        }
    
    print("Created results from manual_tuned_results")
    
else:
    raise RuntimeError(
        "No processed results found. You need to run the main lateral-profile "
        "analysis cell first. The diagnostic Excel plotting cell is not enough."
    )

print("Available keys in results:")
print(results.keys())

for key in results:
    print(f"\n{key}:")
    print("  available fields:", results[key].keys())

## 10. Edge-based x-axis calibration


In [ ]:
# ============================================================
# EDGE-BASED X-AXIS CALIBRATION CELL
# Align simulation to measured using 50% field edges.
# This changes ONLY x-coordinate scale/shift, not dose values.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from pathlib import Path

# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------
edge_level_pct = 50.0          # use 50% field edges / FWHM
save_fig_edge_aligned = True
save_dir = Path("build")
save_name_edge = f"edge_aligned_lateral_profiles_{lateral_mode_select}_{lateral_applicator_select_cm}cm.png"

profile_xlim = (-180, 180)
profile_ylim = (0, 105)
diff_ylim = (-20, 20)
gamma_ylim = (0, 2.0)

# If True, print detailed edge/scale information.
print_alignment_debug = True

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def normalize_to_100(y):
    y = np.asarray(y, dtype=float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax == 0:
        return np.zeros_like(y)
    return 100.0 * y / ymax


def get_left_right_crossings(x, y, level_pct=50.0):
    """
    Return left and right x positions where profile crosses level_pct.
    Uses linear interpolation around the crossing.

    Assumes the profile has one main peak.
    """
    x = np.asarray(x, dtype=float)
    y = normalize_to_100(y)

    # Sort by x just in case
    order = np.argsort(x)
    x = x[order]
    y = y[order]

    if len(x) < 4 or np.nanmax(y) <= 0:
        return np.nan, np.nan

    i_peak = int(np.nanargmax(y))
    level = float(level_pct)

    # Left crossing: moving from peak to left
    x_left = np.nan
    for i in range(i_peak, 0, -1):
        y1 = y[i - 1]
        y2 = y[i]
        if (y1 <= level <= y2) or (y2 <= level <= y1):
            x1 = x[i - 1]
            x2 = x[i]
            if y2 == y1:
                x_left = x1
            else:
                t = (level - y1) / (y2 - y1)
                x_left = x1 + t * (x2 - x1)
            break

    # Right crossing: moving from peak to right
    x_right = np.nan
    for i in range(i_peak, len(y) - 1):
        y1 = y[i]
        y2 = y[i + 1]
        if (y1 >= level >= y2) or (y2 >= level >= y1):
            x1 = x[i]
            x2 = x[i + 1]
            if y2 == y1:
                x_right = x1
            else:
                t = (level - y1) / (y2 - y1)
                x_right = x1 + t * (x2 - x1)
            break

    return float(x_left), float(x_right)


def edge_align_sim_to_measured(meas_x, meas_y, sim_x, sim_y, level_pct=50.0):
    """
    Compute x_scale and x_shift so simulation 50% left/right edges
    match measured 50% left/right edges.

    Returns:
      sim_x_aligned, info dict
    """
    meas_L, meas_R = get_left_right_crossings(meas_x, meas_y, level_pct)
    sim_L, sim_R = get_left_right_crossings(sim_x, sim_y, level_pct)

    if not all(np.isfinite(v) for v in [meas_L, meas_R, sim_L, sim_R]):
        return sim_x.copy(), {
            "ok": False,
            "reason": "failed edge crossing",
            "x_scale": 1.0,
            "x_shift_mm": 0.0,
            "meas_L": meas_L,
            "meas_R": meas_R,
            "sim_L": sim_L,
            "sim_R": sim_R,
            "meas_width": np.nan,
            "sim_width": np.nan,
        }

    meas_width = meas_R - meas_L
    sim_width = sim_R - sim_L

    if sim_width == 0 or not np.isfinite(sim_width):
        return sim_x.copy(), {
            "ok": False,
            "reason": "zero/invalid sim width",
            "x_scale": 1.0,
            "x_shift_mm": 0.0,
            "meas_L": meas_L,
            "meas_R": meas_R,
            "sim_L": sim_L,
            "sim_R": sim_R,
            "meas_width": meas_width,
            "sim_width": sim_width,
        }

    # Scale sim width to measured width
    x_scale = meas_width / sim_width

    # Match centers after scaling
    meas_center = 0.5 * (meas_L + meas_R)
    sim_center = 0.5 * (sim_L + sim_R)
    x_shift = meas_center - x_scale * sim_center

    sim_x_aligned = x_scale * np.asarray(sim_x, dtype=float) + x_shift

    return sim_x_aligned, {
        "ok": True,
        "reason": "ok",
        "x_scale": float(x_scale),
        "x_shift_mm": float(x_shift),
        "meas_L": float(meas_L),
        "meas_R": float(meas_R),
        "sim_L": float(sim_L),
        "sim_R": float(sim_R),
        "meas_width": float(meas_width),
        "sim_width": float(sim_width),
    }


def interp_safe_local(x_src, y_src, x_tgt):
    x_src = np.asarray(x_src, dtype=float)
    y_src = np.asarray(y_src, dtype=float)
    x_tgt = np.asarray(x_tgt, dtype=float)

    order = np.argsort(x_src)
    x_src = x_src[order]
    y_src = y_src[order]

    y = np.interp(x_tgt, x_src, y_src)
    y[(x_tgt < np.nanmin(x_src)) | (x_tgt > np.nanmax(x_src))] = np.nan
    return y


def gamma_1d_local(ref_x, ref_y, eval_x, eval_y, dose_crit=2.0, dist_crit=2.0):
    ref_x = np.asarray(ref_x, dtype=float)
    ref_y = np.asarray(ref_y, dtype=float)
    eval_x = np.asarray(eval_x, dtype=float)
    eval_y = np.asarray(eval_y, dtype=float)

    gamma_vals = np.full_like(ref_x, np.nan, dtype=float)

    for i, (xm, ym) in enumerate(zip(ref_x, ref_y)):
        dist_term = ((eval_x - xm) / dist_crit) ** 2
        dose_term = ((eval_y - ym) / dose_crit) ** 2
        gamma_vals[i] = np.sqrt(np.nanmin(dist_term + dose_term))

    return gamma_vals


def style_axis_local(ax, x_major=None, x_minor=None, y_major=None, y_minor=None):
    ax.tick_params(axis="both", which="major", direction="in", length=5, width=1.0, labelsize=10)
    ax.tick_params(axis="both", which="minor", direction="in", length=2.5, width=0.8)

    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

    if x_major is not None:
        ax.xaxis.set_major_locator(MultipleLocator(x_major))
    if x_minor is not None:
        ax.xaxis.set_minor_locator(MultipleLocator(x_minor))
    else:
        ax.xaxis.set_minor_locator(AutoMinorLocator())

    if y_major is not None:
        ax.yaxis.set_major_locator(MultipleLocator(y_major))
    if y_minor is not None:
        ax.yaxis.set_minor_locator(MultipleLocator(y_minor))
    else:
        ax.yaxis.set_minor_locator(AutoMinorLocator())


# ------------------------------------------------------------
# Process edge-aligned results
# ------------------------------------------------------------
edge_results = {}
edge_summary_rows = []

for key in lateral_profile_depth_keys:
    r = results[key]

    meas_x = np.asarray(r["meas_x"], dtype=float)
    meas_y = np.asarray(r["meas_y"], dtype=float)

    sim_x = np.asarray(r["sim_x"], dtype=float)
    sim_y = np.asarray(r["sim_y"], dtype=float)

    # Align simulation x-axis to measured field edges
    sim_x_aligned, info = edge_align_sim_to_measured(
        meas_x=meas_x,
        meas_y=meas_y,
        sim_x=sim_x,
        sim_y=sim_y,
        level_pct=edge_level_pct
    )

    # Compare on measured grid
    sim_on_meas = interp_safe_local(sim_x_aligned, sim_y, meas_x)

    comp = pd.DataFrame({
        "x_mm": meas_x,
        "measured_pct": meas_y,
        "sim_pct": sim_on_meas
    }).dropna()

    comp["diff_pctpts"] = comp["sim_pct"] - comp["measured_pct"]

    gamma_vals = gamma_1d_local(
        ref_x=comp["x_mm"].values,
        ref_y=comp["measured_pct"].values,
        eval_x=sim_x_aligned,
        eval_y=sim_y,
        dose_crit=gamma_dose_percent,
        dist_crit=gamma_dist_mm
    )

    gamma_pass_rate = 100.0 * np.mean(gamma_vals <= 1.0) if len(gamma_vals) else np.nan

    rmse = float(np.sqrt(np.mean(comp["diff_pctpts"].values ** 2))) if len(comp) else np.nan
    mae = float(np.mean(np.abs(comp["diff_pctpts"].values))) if len(comp) else np.nan
    max_abs = float(np.max(np.abs(comp["diff_pctpts"].values))) if len(comp) else np.nan

    edge_results[key] = {
        "meas_x": meas_x,
        "meas_y": meas_y,
        "sim_x_raw": sim_x,
        "sim_x": sim_x_aligned,
        "sim_y": sim_y,
        "comp": comp,
        "gamma": gamma_vals,
        "align_info": info,
        "RMSE_pctpts": rmse,
        "MAE_pctpts": mae,
        "MaxAbs_pctpts": max_abs,
        "GammaPassRate_pct": gamma_pass_rate,
    }

    edge_summary_rows.append({
        "depth": key,
        "align_ok": info["ok"],
        "x_scale": info["x_scale"],
        "x_shift_mm": info["x_shift_mm"],
        "meas_width_mm": info["meas_width"],
        "sim_width_raw_mm": info["sim_width"],
        "RMSE_pctpts": rmse,
        "MAE_pctpts": mae,
        "MaxAbs_pctpts": max_abs,
        "GammaPassRate_pct": gamma_pass_rate,
    })

edge_summary_df = pd.DataFrame(edge_summary_rows)

print("Edge-alignment summary:")
display(edge_summary_df.round(4))

if print_alignment_debug:
    for key in lateral_profile_depth_keys:
        info = edge_results[key]["align_info"]
        print(f"\n{key}")
        for k, v in info.items():
            print(f"  {k}: {v}")


# ------------------------------------------------------------
# Plot edge-aligned figure
# ------------------------------------------------------------
fig, axes = plt.subplots(
    2, 3,
    figsize=(16, 7.2),
    sharex="col",
    gridspec_kw={"height_ratios": [2.6, 0.9]}
)

fig.subplots_adjust(wspace=0.38, hspace=0.10)

pretty_names = {
    "dmax": "dmax",
    "r50": "R50",
    "middepth": "mid-depth"
}

for col, key in enumerate(lateral_profile_depth_keys):
    r = edge_results[key]

    ax_top = axes[0, col]
    ax_mid = axes[1, col]
    ax_gamma = ax_top.twinx()

    # Top: profile overlay
    ax_top.plot(
        r["meas_x"],
        r["meas_y"],
        "-",
        lw=3.2,
        color="#0072B2",
        label="Measured"
    )

    ax_top.plot(
        r["sim_x"],
        r["sim_y"],
        "--",
        lw=3.0,
        color="#D55E00",
        label="Simulation edge-aligned"
    )

    # Optional: show raw sim lightly so you can see what changed
    ax_top.plot(
        r["sim_x_raw"],
        r["sim_y"],
        ":",
        lw=1.4,
        color="#D55E00",
        alpha=0.45,
        label="Simulation raw x"
    )

    ax_top.axvline(0, color="gray", lw=1.0, ls="--", alpha=0.7)
    ax_top.set_title(
        f"{pretty_names[key]} | {lateral_mode_select} {lateral_applicator_select_cm} cm",
        fontsize=12
    )
    ax_top.set_ylabel("Relative dose (%)", fontsize=11)
    ax_top.set_xlim(*profile_xlim)
    ax_top.set_ylim(*profile_ylim)
    style_axis_local(ax_top, x_major=100, x_minor=50, y_major=20, y_minor=10)
    for spine in ax_top.spines.values():
        spine.set_color("black")
    # The profile's right spine is removed so it cannot overlap the gamma axis.
    ax_top.spines["right"].set_visible(False)

    # Gamma is overlaid on the relative-dose profile using the right y-axis.
    ax_gamma.axhline(1.0, color="black", lw=1.5, ls="--")
    gamma_line, = ax_gamma.plot(
        r["comp"]["x_mm"].values,
        r["gamma"],
        "-",
        lw=2.0,
        color=color_gamma,
        label="Gamma 2% / 2 mm",
    )
    ax_gamma.set_ylim(*gamma_ylim)
    ax_gamma.set_ylabel("Gamma index", fontsize=11, color=color_gamma)
    ax_gamma.tick_params(axis="y", which="both", colors=color_gamma, direction="in")
    ax_gamma.spines["right"].set_color(color_gamma)
    ax_gamma.spines["right"].set_linewidth(1.2)
    ax_gamma.spines["top"].set_visible(False)
    ax_gamma.spines["bottom"].set_visible(False)
    ax_gamma.spines["left"].set_visible(False)

    profile_handles, profile_labels = ax_top.get_legend_handles_labels()
    ax_top.legend(
        profile_handles + [gamma_line],
        profile_labels + [gamma_line.get_label()],
        frameon=False,
        fontsize=8,
        loc="best",
    )

    # Bottom: difference
    comp = r["comp"]

    ax_mid.axhline(0, color="black", lw=1.5)
    ax_mid.plot(
        comp["x_mm"].values,
        comp["diff_pctpts"].values,
        "-",
        lw=2.0,
        color="#0072B2"
    )

    ax_mid.set_ylabel("Diff. (pp)", fontsize=11)
    ax_mid.set_xlim(*profile_xlim)
    ax_mid.set_ylim(*diff_ylim)
    ax_mid.set_xlabel("Centered position (mm)", fontsize=11)
    style_axis_local(ax_mid, x_major=100, x_minor=50, y_major=5, y_minor=1)
    for spine in ax_mid.spines.values():
        spine.set_color("black")

plt.tight_layout()

if save_fig_edge_aligned:
    save_dir.mkdir(parents=True, exist_ok=True)
    outpath = save_dir / save_name_edge
    fig.savefig(
        outpath,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none"
    )
    print(f"Saved edge-aligned figure: {outpath}")

plt.show()

## 11. Final measured-vs-simulation analysis

This consolidated cell produces the profile overlays, percent-difference plots, continuous gamma plots, and match summary. Superseded duplicate tuning/plotting cells were removed during cleanup.


In [ ]:
# ============================================================
# SELF-CONTAINED COMBINED CELL:
# MEASURED vs SIMULATION
# + TAIL X-TUNING
# + TOP/SHOULDER Y-TUNING
# + % DIFFERENCE
# + SMOOTHED CONTINUOUS GAMMA
# + MATCH SUMMARY
# ------------------------------------------------------------
# Paste this AFTER your edge_results/results creation cell.
# This cell does NOT import local files, so no file-not-found issue.
# ============================================================

import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import MaxNLocator, AutoMinorLocator

# ============================================================
# PICK INPUT SOURCE
# ============================================================

if "edge_results" in globals():
    source_results = edge_results
    source_kind = "edge_results"
elif "results" in globals():
    source_results = results
    source_kind = "results"
elif "manual_tuned_results" in globals():
    source_results = manual_tuned_results
    source_kind = "manual_tuned_results"
elif "profile_diff_gamma_results" in globals():
    source_results = profile_diff_gamma_results
    source_kind = "profile_diff_gamma_results"
else:
    raise RuntimeError(
        "No usable source found. Run your fresh results creation cell and edge-alignment cell first."
    )

print(f"Using source: {source_kind}")

depth_keys = list(lateral_profile_depth_keys)

is_flash9_2cm_lateral_profile = (
    str(measured_lateral_excel_sheet).lower() == "flash9"
    and float(lateral_applicator_select_cm) == 2.0
)

# ============================================================
# SAVE SETTINGS
# ============================================================

save_fig = True
save_dir = Path("build")

def _safe_filename_part(value):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_")


def infer_energy_label():
    if "lateral_energy_mev" in globals():
        return f"{lateral_energy_mev:g} MeV"

    for value in (
        globals().get("measured_lateral_excel_sheet", ""),
        globals().get("lateral_mode_select", ""),
    ):
        match = re.search(r"(\d+(?:\.\d+)?)", str(value))
        if match:
            return f"{float(match.group(1)):g} MeV"

    return "Energy not specified"


energy_label = infer_energy_label()
energy_file_label = _safe_filename_part(energy_label.replace(" ", ""))
diameter_file_label = f"{lateral_applicator_select_cm:g}cm"
mode_file_label = _safe_filename_part(lateral_mode_select)

save_base_name = (
    f"lateral_profiles_{energy_file_label}_{diameter_file_label}_"
    f"{mode_file_label}_auto-match"
)

save_png = True
save_pdf = True
save_tiff = True
save_dpi = 300

# ============================================================
# GAMMA / DIFFERENCE CONTROLS
# ============================================================

gamma_dose_percent = 2.0
gamma_dist_mm = 2.0
gamma_dose_mode = "global"

gamma_summary_dose_threshold_pct = 10.0

gamma_search_radius_mm = 4.0 * gamma_dist_mm
gamma_search_step_mm = 0.10

diff_ylim = (-3, 3)
gamma_ylim = (0, 2.0)
profile_ylim = (0, 105)

# ============================================================
# PLOT CONTROLS
# ============================================================

# Use zero x padding so every subplot starts/ends exactly at the measured profile range.
x_padding_mm = 0.0
x_padding_fraction = 0.0

show_center_line = True
show_title_tuning_values = True
show_legend = True

plt.rcParams.update({
    "axes.grid": False,
    "axes.linewidth": 1.25,
    "font.size": 13,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "figure.dpi": 120,
    "savefig.dpi": save_dpi,
})

title_fontsize = 16
axis_label_fontsize = 14
tick_labelsize = 12
legend_fontsize = 11

# Color-blind friendly colors
color_meas = "#0072B2"
color_sim = "#E69F00"
color_diff = "#009E73"
color_gamma = "#CC79A7"

lw_meas = 3.8
lw_sim = 3.9
lw_secondary = 2.6
lw_ref = 1.4

pretty_names = {
    "dmax": "Dmax",
    "r50": "R50",
    "middepth": "Mid-depth",
}

# ------------------------------------------------------------
# IMPORTANT GAMMA SMOOTHING CONTROLS
# ------------------------------------------------------------
# If gamma is too noisy / spiky, increase these:
#   gamma_profile_smooth_window_mm: smooths dose curves BEFORE gamma
#   gamma_display_smooth_window_mm: smooths gamma curve AFTER gamma
#
# Suggested values:
#   mild:   1.0 to 1.5
#   medium: 2.0 to 2.5
#   strong: 3.0 to 4.0
# ------------------------------------------------------------

use_smoothed_profiles_for_gamma = True
gamma_profile_smooth_window_mm = 2.0

# Plot raw/unsmoothed gamma for every profile.
smooth_gamma_for_display = False
gamma_display_smooth_window_mm = 1.2

# If True, Gamma Pass uses the smoothed gamma line.
# If False, Gamma Pass uses raw gamma.
use_smoothed_gamma_for_summary = True

# Plot raw/unsmoothed percentage-difference curves for every profile.
smooth_difference_for_display = False
difference_smooth_window_mm = 1.0

# ============================================================
# BASELINE + DISPLAY SMOOTHING
# ============================================================

baseline_correct_measured = True
baseline_correct_simulation = True

baseline_edge_fraction = 0.10
baseline_percentile = 0.0

smooth_measured_profile_for_display = True
smooth_sim_profile_for_display = True

measured_profile_smooth_window_mm = 4.0
sim_profile_smooth_window_mm = 4.0

# Keep False for real difference metrics.
# Gamma has its own smoothing controls above.
use_smoothed_profiles_for_difference = False

# ============================================================
# OPTIONAL SHAPE BLEND
# ============================================================
# 0% = do not blend simulation to measured.
# 100% = fully force simulation onto measured.
#
# I recommend keeping this LOW or 0 for real validation.
# Use it only as a diagnostic knob to see how much shape mismatch remains.
# ============================================================

# Optional diagnostic shape blend percentage by depth.
# Keep all values at 0.0 for validation so the simulation is not forced
# toward the measured profile. Increase a value only as a diagnostic knob.
shape_match_percent_by_depth = {
    "dmax": 0.0,
    "r50": 0.0,
    "middepth": 0.0,
}

flip_simulation_x_by_depth = {
    "dmax": True,
    "r50": False,
    "middepth": True,
}

# ============================================================
# SIMULATION X-TAIL / PENUMBRA TUNING
# ============================================================
# left_shift_max_mm:
#   positive = move left tail/penumbra right
#   negative = move left tail/penumbra left
#
# right_shift_max_mm:
#   positive = move right tail/penumbra right
#   negative = move right tail/penumbra left
#
# blend_width_mm:
#   larger = softer / smoother x correction
#   smaller = pointier / sharper x correction
#
# power:
#   smaller = softer
#   larger = pointier
# ============================================================

flip_simulation_x_by_depth = {
    "dmax": True,
    "r50": False,
    "middepth": True,
}

sim_x_tail_tune = {
    "dmax": {
        "x_scale": 1.000,
        "x_shift_mm": 0.0,

        # 5 cm dmax: shoulders/edge around +/-50 mm.
        # Slight widening only.
        "left_start_mm": -48.0,
        "left_shift_max_mm": -1.5,

        "right_start_mm": 48.0,
        "right_shift_max_mm": -3,

        "blend_width_mm": 24.0,
        "power": 1.5,
    },

    "r50": {
        "x_scale": 1.000,
        "x_shift_mm": 0.0,

        # R50 gamma failure is mainly left shoulder/plateau and right shoulder.
        # Slight inward correction to reduce the shoulder mismatch.
        "left_start_mm": -50.0,
        "left_shift_max_mm": 1.5,

        "right_start_mm": 50.0,
        "right_shift_max_mm": -1.5,

        "blend_width_mm": 28.0,
        "power": 1.4,
    },

    "middepth": {
        "x_scale": 1.000,
        "x_shift_mm": 0.0,

        # Middepth has strong gamma around -40 and +35.
        # Slight inward correction helps both shoulder gamma peaks.
        "left_start_mm": -48.0,
        "left_shift_max_mm": 2.0,

        "right_start_mm": 48.0,
        "right_shift_max_mm": -2.0,

        "blend_width_mm": 28.0,
        "power": 1.4,
    },
}
# ============================================================
# SIMULATION TOP / SHOULDER Y-SHAPE TUNING
# ============================================================
# IMPORTANT:
#   dmax top tuning is disabled because it was hurting gamma.
#
# center_adjust_pct:
#   positive raises center plateau
#   negative lowers center plateau
#
# left/right_shoulder_adjust_pct:
#   positive raises shoulder
#   negative lowers shoulder
#
# top_smooth_window_mm:
#   larger = softer top
#   smaller = pointier top
#
# shoulder_width_mm:
#   larger = broader/softer shoulder adjustment
#   smaller = localized/pointier shoulder adjustment
#
# shoulder_power:
#   smaller = softer
#   larger = pointier
# ============================================================

sim_top_shape_tune = {
    "dmax": {
        "enabled": True,

        # Dmax already close. Keep correction gentle.
        "top_smooth_window_mm": 5.0,
        "top_smooth_threshold_pct": 78.0,
        "top_smooth_blend_pct": 12.0,

        "center_x_mm": 0.0,
        "center_width_mm": 45.0,
        "center_adjust_pct": 0.25,

        # 5 cm shoulder positions.
        "left_shoulder_x_mm": -42.0,
        "right_shoulder_x_mm": 42.0,

        # Dmax gamma is already 95. Do not overcorrect.
        "left_shoulder_adjust_pct": -0.8,
        "right_shoulder_adjust_pct": -1.0,

        "shoulder_width_mm": 13.0,
        "shoulder_power": 2.0,

        "renormalize_to_100": True,
    },

    "r50": {
        "enabled": True,

        # 20 mm smoothing is too much for 5 cm.
        "top_smooth_window_mm": 10.0,
        "top_smooth_threshold_pct": 78.0,
        "top_smooth_blend_pct": 12.0,

        # In your plot, R50 simulation is high on the left/top region.
        # Slightly lower broad center-left.
        "center_x_mm": -40.0,
        "center_width_mm": 50.0,
        "center_adjust_pct": -0.35,

        # 5 cm shoulders are around +/-45 to +/-50.
        "left_shoulder_x_mm": -45.0,
        "right_shoulder_x_mm": 45.0,

        # Lower both shoulders, stronger on left.
        "left_shoulder_adjust_pct": -4,
        "right_shoulder_adjust_pct": -1.2,

        "shoulder_width_mm": 16.0,
        "shoulder_power": 1.8,

        "renormalize_to_100": True,
    },

    "middepth": {
        "enabled": True,

        # Middepth needs smoother broad correction but not the 10 cm values.
        "top_smooth_window_mm": 20.0,
        "top_smooth_threshold_pct": 78.0,
        "top_smooth_blend_pct": 12.0,

        # The plot shows:
        #   negative diff around -45 mm => simulation too low there
        #   positive diff around +30/+40 mm => simulation too high there
        # So raise left-center and lower right-center.
        "center_x_mm": -10.0,
        "center_width_mm": 38.0,
        "center_adjust_pct": 1.2,

        "left_shoulder_x_mm": -20.0,
        "right_shoulder_x_mm": 30.0,

        # Left side needs raising, right side needs lowering.
        "left_shoulder_adjust_pct":-4,
        "right_shoulder_adjust_pct": -2.8,

        "shoulder_width_mm": 18.0,
        "shoulder_power": 1.7,

        "renormalize_to_100": True,
    },
}

# Extra head/tip sharpening only for FLASH 9 MeV 2 cm lateral profiles.
# These values still only modify the simulation curve. Everything else keeps
# the older/general settings because those already worked better there.
if is_flash9_2cm_lateral_profile:
    sim_top_shape_tune["dmax"].update({
        "top_smooth_window_mm": 0.0,
        "center_x_mm": 0.0,
        "center_width_mm": 18.0,
        "center_adjust_pct": 1.8,
        "left_shoulder_x_mm": -16.0,
        "right_shoulder_x_mm": 16.0,
        "left_shoulder_adjust_pct": 0.8,
        "right_shoulder_adjust_pct": 0.8,
        "shoulder_width_mm": 8.0,
        "shoulder_power": 2.5,
    })
    sim_top_shape_tune["middepth"].update({
        "top_smooth_window_mm": 0.0,
        "center_x_mm": 0.0,
        "center_width_mm": 20.0,
        "center_adjust_pct": 2.2,
        "left_shoulder_x_mm": -17.0,
        "right_shoulder_x_mm": 17.0,
        "left_shoulder_adjust_pct": 1.2,
        "right_shoulder_adjust_pct": 1.2,
        "shoulder_width_mm": 8.0,
        "shoulder_power": 2.5,
    })

# ============================================================
# OPTIONAL AUTOMATIC SIMULATION SHAPE MATCHING
# ============================================================
# This adjusts ONLY the simulation dose curve, never the measured curve.
# Instead of tuning separate numbers for dmax/R50/mid-depth, choose one simple
# matching level and one amount. The notebook automatically estimates the
# correction needed at each depth from the measured-minus-simulation residual.
#
# auto_sim_shape_match_level options:
#   "off"    = no automatic matching
#   "low"    = small, broad correction
#   "medium" = balanced correction
#   "high"   = stronger correction, still smoothed/capped
#
# auto_sim_shape_match_amount:
#   0.0 = no correction, 1.0 = full selected preset strength.
#   Use this as the main single knob if you want less/more matching.
#
# Head/penumbra matching:
#   The y-shape auto-match now gives extra weight to the high-dose head
#   and the 20-80% penumbra region so those regions follow measured data
#   more closely without manually tuning dmax/R50/mid-depth one by one.
# ============================================================

auto_sim_shape_match_enabled = True

# One simple global knob for automatic matching.
# It controls BOTH automatic tail/penumbra x-matching and automatic head/shape y-matching.
# Automatic matching controls the simulation head, penumbra, and tails.
# Keep this high/0.80 because that setting gave the best automatic agreement.
auto_sim_match_level = "high"
auto_sim_match_amount = 0.80

# Prefer automatic matching instead of hand-tuned values.
# When False, the notebook skips the manual x-tail and y-shape dictionaries
# except for the special flash9/2 cm head-tip sharpening block.
use_manual_tuning_before_auto_match = False

# Backward-compatible names used by the helper below.
auto_sim_shape_match_level = auto_sim_match_level
auto_sim_shape_match_amount = auto_sim_match_amount

auto_sim_x_tail_match_enabled = True
auto_sim_x_tail_match_presets = {
    "off": {"enabled": False, "search_mm": 0.0, "step_mm": 1.0, "scale_range": 0.0, "dose_threshold_pct": 10.0},
    "low": {"enabled": True, "search_mm": 3.0, "step_mm": 1.0, "scale_range": 0.01, "dose_threshold_pct": 10.0, "tail_level_pct": 50.0},
    "medium": {"enabled": True, "search_mm": 7.0, "step_mm": 1.0, "scale_range": 0.02, "dose_threshold_pct": 10.0, "tail_level_pct": 50.0},
    "high": {"enabled": True, "search_mm": 16.0, "step_mm": 1.0, "scale_range": 0.08, "dose_threshold_pct": 5.0, "tail_level_pct": 50.0},
}

auto_sim_shape_match_presets = {
    "off": {
        "enabled": False,
        "strength": 0.0,
        "residual_cap_fraction": 0.0,
        "smooth_window_mm": 30.0,
        "min_measured_dose_pct": 15.0,
        "min_adjust_pct": 0.0,
        "max_adjust_pct": 0.0,
        "renormalize_to_100": True,
    },
    "low": {
        "enabled": True,
        "strength": 0.25,
        "residual_cap_fraction": 0.35,
        "smooth_window_mm": 30.0,
        "min_measured_dose_pct": 15.0,
        "min_adjust_pct": 1.0,
        "max_adjust_pct": 2.5,
        "head_weight": 1.0,
        "penumbra_weight": 1.2,
        "renormalize_to_100": True,
    },
    "medium": {
        "enabled": True,
        "strength": 0.45,
        "residual_cap_fraction": 0.55,
        "smooth_window_mm": 22.0,
        "min_measured_dose_pct": 15.0,
        "min_adjust_pct": 1.5,
        "max_adjust_pct": 4.0,
        "head_weight": 1.1,
        "penumbra_weight": 1.5,
        "renormalize_to_100": True,
    },
    "high": {
        "enabled": True,
        "strength": 0.90,
        "residual_cap_fraction": 1.00,
        "smooth_window_mm": 10.0,
        "min_measured_dose_pct": 5.0,
        "min_adjust_pct": 2.0,
        "max_adjust_pct": 10.0,
        "head_weight": 2.5,
        "penumbra_weight": 1.8,
        "renormalize_to_100": True,
    },
}

# Optional advanced overrides if one depth needs special handling.
# Leave this empty for fully automatic dmax/R50/mid-depth matching.
auto_sim_shape_match_depth_overrides = {}


def get_auto_sim_x_tail_match_settings(depth_key):
    level = str(auto_sim_match_level).lower()
    if level not in auto_sim_x_tail_match_presets:
        raise ValueError(
            f"Unknown auto_sim_match_level={auto_sim_match_level!r}. "
            f"Choose one of {list(auto_sim_x_tail_match_presets)}."
        )

    settings = auto_sim_x_tail_match_presets[level].copy()
    amount = float(np.clip(auto_sim_match_amount, 0.0, 1.0))
    settings["enabled"] = bool(auto_sim_x_tail_match_enabled and settings.get("enabled", False) and amount > 0.0)
    settings["search_mm"] = float(settings.get("search_mm", 0.0)) * amount
    settings["scale_range"] = float(settings.get("scale_range", 0.0)) * amount
    settings["level"] = level
    settings["amount"] = amount

    return settings


def get_auto_sim_shape_match_settings(depth_key):
    level = str(auto_sim_shape_match_level).lower()
    if level not in auto_sim_shape_match_presets:
        raise ValueError(
            f"Unknown auto_sim_shape_match_level={auto_sim_shape_match_level!r}. "
            f"Choose one of {list(auto_sim_shape_match_presets)}."
        )

    settings = auto_sim_shape_match_presets[level].copy()
    settings.update(auto_sim_shape_match_depth_overrides.get(depth_key, {}))

    amount = float(np.clip(auto_sim_shape_match_amount, 0.0, 1.0))
    settings["enabled"] = bool(auto_sim_shape_match_enabled and settings.get("enabled", False) and amount > 0.0)
    settings["strength"] = float(settings.get("strength", 0.0)) * amount
    settings["residual_cap_fraction"] = float(settings.get("residual_cap_fraction", 0.0)) * amount
    settings["level"] = level
    settings["amount"] = amount

    return settings


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_array_from_result(r, names):
    for name in names:
        if name in r:
            return np.asarray(r[name], dtype=float)
    raise KeyError(f"Could not find any of these keys: {names}")


def interp_safe(x_src, y_src, x_tgt):
    x_src = np.asarray(x_src, dtype=float)
    y_src = np.asarray(y_src, dtype=float)
    x_tgt = np.asarray(x_tgt, dtype=float)

    order = np.argsort(x_src)
    x_src = x_src[order]
    y_src = y_src[order]

    y = np.interp(x_tgt, x_src, y_src)
    y[(x_tgt < np.nanmin(x_src)) | (x_tgt > np.nanmax(x_src))] = np.nan

    return y


def moving_average_reflect(y, win_pts):
    y = np.asarray(y, dtype=float)

    if win_pts <= 1 or len(y) < 3:
        return y.copy()

    win_pts = int(win_pts)

    if win_pts % 2 == 0:
        win_pts += 1

    pad = win_pts // 2
    ypad = np.pad(y, (pad, pad), mode="reflect")
    kernel = np.ones(win_pts) / win_pts

    return np.convolve(ypad, kernel, mode="valid")


def smooth_by_mm_y_only(x, y, window_mm):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if window_mm is None or window_mm <= 0 or len(x) < 3:
        return y.copy()

    dx = np.nanmedian(np.diff(np.sort(x)))

    if not np.isfinite(dx) or dx <= 0:
        return y.copy()

    finite = np.isfinite(y)
    if not np.any(finite):
        return y.copy()

    y_filled = y.copy()
    if not np.all(finite):
        y_filled[~finite] = np.interp(x[~finite], x[finite], y[finite])

    win_pts = max(1, int(round(window_mm / dx)))

    return moving_average_reflect(y_filled, win_pts)


def smooth_by_mm(x, y, window_mm):
    return np.asarray(x, dtype=float).copy(), smooth_by_mm_y_only(x, y, window_mm)


def baseline_to_zero(y, edge_fraction=0.10, percentile=0.0):
    y = np.asarray(y, dtype=float)

    if len(y) < 4:
        return y.copy(), np.nan

    n_edge = max(1, int(round(edge_fraction * len(y))))

    edge_values = np.concatenate([
        y[:n_edge],
        y[-n_edge:],
    ])

    edge_values = edge_values[np.isfinite(edge_values)]

    if len(edge_values) == 0:
        return y.copy(), np.nan

    baseline = float(np.nanpercentile(edge_values, percentile))

    y0 = y - baseline
    y0 = np.clip(y0, 0.0, None)

    ymax = np.nanmax(y0)

    if np.isfinite(ymax) and ymax > 0:
        y0 = 100.0 * y0 / ymax

    return y0, baseline


def force_profile_tails_to_zero(x, y, n_edge_points=1):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float).copy()

    if len(y) == 0 or n_edge_points <= 0:
        return y

    n = min(int(n_edge_points), len(y) // 2 if len(y) > 1 else 1)
    y[:n] = 0.0
    y[-n:] = 0.0

    ymax = np.nanmax(y)
    if np.isfinite(ymax) and ymax > 0:
        y = 100.0 * y / ymax

    return y


force_profile_endpoints_to_zero = True
endpoint_zero_points = 1


def get_xlim_from_measured(meas_x, pad_mm=4.0, pad_frac=0.04):
    meas_x = np.asarray(meas_x, dtype=float)
    meas_x = meas_x[np.isfinite(meas_x)]

    xmin = float(np.nanmin(meas_x))
    xmax = float(np.nanmax(meas_x))
    width = xmax - xmin

    pad = max(pad_mm, pad_frac * width)

    return xmin - pad, xmax + pad


def apply_sim_flip(sim_x, flip=False):
    sim_x = np.asarray(sim_x, dtype=float)

    if flip:
        return -1.0 * sim_x

    return sim_x.copy()


def apply_side_x_tune(sim_x, tune):
    sim_x = np.asarray(sim_x, dtype=float)

    x_scale = float(tune.get("x_scale", 1.0))
    x_shift = float(tune.get("x_shift_mm", 0.0))

    x = x_scale * sim_x + x_shift
    x_tuned = x.copy()

    left_start = float(tune.get("left_start_mm", -np.inf))
    right_start = float(tune.get("right_start_mm", np.inf))

    left_shift_max = float(tune.get("left_shift_max_mm", 0.0))
    right_shift_max = float(tune.get("right_shift_max_mm", 0.0))

    blend_width = float(tune.get("blend_width_mm", 15.0))
    power = float(tune.get("power", 2.0))

    if not np.isfinite(blend_width) or blend_width <= 0:
        blend_width = 15.0

    if not np.isfinite(power) or power <= 0:
        power = 2.0

    # Left side
    left_mask = x < left_start
    if np.any(left_mask) and left_shift_max != 0.0:
        u_left = (left_start - x[left_mask]) / blend_width
        u_left = np.clip(u_left, 0.0, 5.0)
        ramp_left = 1.0 - np.exp(-(u_left ** power))
        x_tuned[left_mask] = x[left_mask] + left_shift_max * ramp_left

    # Right side
    right_mask = x > right_start
    if np.any(right_mask) and right_shift_max != 0.0:
        u_right = (x[right_mask] - right_start) / blend_width
        u_right = np.clip(u_right, 0.0, 5.0)
        ramp_right = 1.0 - np.exp(-(u_right ** power))
        x_tuned[right_mask] = x[right_mask] + right_shift_max * ramp_right

    return x_tuned


def smoothstep(u):
    u = np.clip(u, 0.0, 1.0)
    return u * u * (3.0 - 2.0 * u)


def shape_weight(x, x0, width, power):
    x = np.asarray(x, dtype=float)

    if width is None or width <= 0:
        return np.zeros_like(x)

    u = np.abs((x - x0) / width)
    u = np.clip(u, 0.0, 20.0)

    return np.exp(-(u ** power))


def apply_sim_top_shape_tune(x, y, tune):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if not tune.get("enabled", False):
        return y.copy()

    y_out = y.copy()

    # High-dose/top smoothing
    top_smooth_window_mm = float(tune.get("top_smooth_window_mm", 0.0))
    top_threshold = float(tune.get("top_smooth_threshold_pct", 75.0))
    top_blend = float(tune.get("top_smooth_blend_pct", 15.0))

    if top_smooth_window_mm > 0:
        y_smooth = smooth_by_mm_y_only(x, y_out, top_smooth_window_mm)

        if top_blend <= 0:
            top_weight = (y_out >= top_threshold).astype(float)
        else:
            top_weight = smoothstep((y_out - top_threshold) / top_blend)

        y_out = top_weight * y_smooth + (1.0 - top_weight) * y_out

    # Center plateau adjustment
    center_x = float(tune.get("center_x_mm", 0.0))
    center_width = float(tune.get("center_width_mm", 20.0))
    center_adjust = float(tune.get("center_adjust_pct", 0.0))

    if center_adjust != 0.0:
        center_w = shape_weight(x, center_x, center_width, 2.0)
        high_w = smoothstep((y_out - 70.0) / 20.0)
        y_out = y_out + center_adjust * center_w * high_w

    # Shoulder adjustment
    left_x = float(tune.get("left_shoulder_x_mm", -20.0))
    right_x = float(tune.get("right_shoulder_x_mm", 20.0))

    left_adjust = float(tune.get("left_shoulder_adjust_pct", 0.0))
    right_adjust = float(tune.get("right_shoulder_adjust_pct", 0.0))

    shoulder_width = float(tune.get("shoulder_width_mm", 10.0))
    shoulder_power = float(tune.get("shoulder_power", 2.0))

    high_w = smoothstep((y_out - 65.0) / 25.0)

    if left_adjust != 0.0:
        left_w = shape_weight(x, left_x, shoulder_width, shoulder_power)
        y_out = y_out + left_adjust * left_w * high_w

    if right_adjust != 0.0:
        right_w = shape_weight(x, right_x, shoulder_width, shoulder_power)
        y_out = y_out + right_adjust * right_w * high_w

    y_out = np.clip(y_out, 0.0, None)

    if bool(tune.get("renormalize_to_100", True)):
        ymax = np.nanmax(y_out)
        if np.isfinite(ymax) and ymax > 0:
            y_out = 100.0 * y_out / ymax

    return y_out


def auto_tune_sim_x_to_measured(meas_x, meas_y, sim_x, sim_y, base_tune, settings):
    info = {
        "enabled": False,
        "level": settings.get("level", "off"),
        "amount": settings.get("amount", 0.0),
        "x_scale_delta": 0.0,
        "left_extra_shift_mm": 0.0,
        "right_extra_shift_mm": 0.0,
        "rmse_before": np.nan,
        "rmse_after": np.nan,
    }

    if not settings.get("enabled", False):
        return sim_x.copy(), info

    meas_x = np.asarray(meas_x, dtype=float)
    meas_y = np.asarray(meas_y, dtype=float)
    sim_x = np.asarray(sim_x, dtype=float)
    sim_y = np.asarray(sim_y, dtype=float)

    dose_threshold = float(settings.get("dose_threshold_pct", 10.0))
    mask = np.isfinite(meas_x) & np.isfinite(meas_y) & (meas_y >= dose_threshold)

    if not np.any(mask):
        return sim_x.copy(), info

    search_mm = float(settings.get("search_mm", 0.0))
    step_mm = float(settings.get("step_mm", 1.0))
    scale_range = float(settings.get("scale_range", 0.0))
    tail_level = float(settings.get("tail_level_pct", 50.0))

    meas_left, meas_right = get_left_right_crossings(meas_x, meas_y, tail_level)
    if not np.isfinite(meas_left) or not np.isfinite(meas_right):
        meas_left = float(np.nanpercentile(meas_x[mask], 10.0))
        meas_right = float(np.nanpercentile(meas_x[mask], 90.0))

    shifts = np.arange(-search_mm, search_mm + 0.5 * step_mm, step_mm) if search_mm > 0 else np.array([0.0])
    scales = np.linspace(1.0 - scale_range, 1.0 + scale_range, 5) if scale_range > 0 else np.array([1.0])

    base_interp = interp_safe(sim_x, sim_y, meas_x)
    ok_base = mask & np.isfinite(base_interp)
    if np.any(ok_base):
        info["rmse_before"] = float(np.sqrt(np.mean((base_interp[ok_base] - meas_y[ok_base]) ** 2)))

    best_rmse = np.inf
    best_x = sim_x.copy()
    best_scale = 1.0
    best_left = 0.0
    best_right = 0.0

    for scale in scales:
        for left_shift in shifts:
            for right_shift in shifts:
                trial_tune = dict(base_tune)
                # sim_x is already manually tuned; these are automatic extra corrections.
                trial_tune["x_scale"] = scale
                trial_tune["x_shift_mm"] = 0.0
                trial_tune["left_start_mm"] = meas_left
                trial_tune["right_start_mm"] = meas_right
                trial_tune["left_shift_max_mm"] = left_shift
                trial_tune["right_shift_max_mm"] = right_shift

                trial_x = apply_side_x_tune(sim_x, trial_tune)
                sim_on_meas = interp_safe(trial_x, sim_y, meas_x)
                ok = mask & np.isfinite(sim_on_meas)
                if not np.any(ok):
                    continue

                residual = sim_on_meas[ok] - meas_y[ok]
                rmse = float(np.sqrt(np.mean(residual ** 2)))

                if rmse < best_rmse:
                    best_rmse = rmse
                    best_x = trial_x
                    best_scale = scale
                    best_left = left_shift
                    best_right = right_shift

    if np.isfinite(best_rmse):
        info.update({
            "enabled": True,
            "x_scale_delta": float(best_scale - 1.0),
            "left_extra_shift_mm": float(best_left),
            "right_extra_shift_mm": float(best_right),
            "rmse_after": float(best_rmse),
        })
        return best_x, info

    return sim_x.copy(), info


def auto_match_sim_profile_to_measured(meas_x, meas_y, sim_x, sim_y, settings):
    meas_x = np.asarray(meas_x, dtype=float)
    meas_y = np.asarray(meas_y, dtype=float)
    sim_x = np.asarray(sim_x, dtype=float)
    sim_y = np.asarray(sim_y, dtype=float)

    info = {
        "enabled": False,
        "strength": 0.0,
        "level": settings.get("level", "off"),
        "amount": settings.get("amount", 0.0),
        "max_adjust_pct": 0.0,
        "smooth_window_mm": 0.0,
        "applied_points": 0,
        "mean_abs_correction_pct": 0.0,
        "max_abs_correction_pct": 0.0,
    }

    if not auto_sim_shape_match_enabled or not settings.get("enabled", False):
        return sim_y.copy(), info

    strength = float(settings.get("strength", 0.0))
    smooth_window = float(settings.get("smooth_window_mm", 20.0))
    min_dose = float(settings.get("min_measured_dose_pct", 15.0))
    residual_cap_fraction = float(settings.get("residual_cap_fraction", 0.0))

    if strength <= 0.0 or residual_cap_fraction <= 0.0:
        return sim_y.copy(), info

    meas_on_sim = interp_safe(meas_x, meas_y, sim_x)
    valid = np.isfinite(meas_on_sim) & np.isfinite(sim_y) & (meas_on_sim >= min_dose)

    if not np.any(valid):
        return sim_y.copy(), info

    residual = np.zeros_like(sim_y, dtype=float)
    residual[valid] = meas_on_sim[valid] - sim_y[valid]

    residual_scale = float(np.nanpercentile(np.abs(residual[valid]), 90.0))
    min_adjust = float(settings.get("min_adjust_pct", 0.0))
    max_adjust_limit = float(settings.get("max_adjust_pct", np.inf))
    max_adjust = residual_cap_fraction * residual_scale
    max_adjust = float(np.clip(max_adjust, min_adjust, max_adjust_limit))

    if max_adjust <= 0.0:
        return sim_y.copy(), info

    smooth_residual = smooth_by_mm_y_only(sim_x, residual, smooth_window)

    dose_weight = np.zeros_like(sim_y, dtype=float)
    dose_weight[valid] = smoothstep((meas_on_sim[valid] - min_dose) / max(100.0 - min_dose, 1.0))

    head_weight = float(settings.get("head_weight", 1.0))
    penumbra_weight = float(settings.get("penumbra_weight", 1.0))

    head_band = np.zeros_like(sim_y, dtype=float)
    head_band[valid] = smoothstep((meas_on_sim[valid] - 80.0) / 15.0)

    penumbra_band = np.zeros_like(sim_y, dtype=float)
    penumbra_band[valid] = (
        smoothstep((meas_on_sim[valid] - 20.0) / 20.0)
        * smoothstep((80.0 - meas_on_sim[valid]) / 20.0)
    )

    region_weight = np.maximum.reduce([
        dose_weight,
        head_weight * head_band,
        penumbra_weight * penumbra_band,
    ])
    region_weight = np.clip(region_weight, 0.0, max(head_weight, penumbra_weight, 1.0))

    correction = strength * smooth_residual * region_weight
    correction[~valid] = 0.0
    correction = np.clip(correction, -max_adjust, max_adjust)

    y_out = np.clip(sim_y + correction, 0.0, None)

    if bool(settings.get("renormalize_to_100", True)):
        ymax = np.nanmax(y_out)
        if np.isfinite(ymax) and ymax > 0:
            y_out = 100.0 * y_out / ymax

    info.update({
        "enabled": True,
        "strength": strength,
        "level": settings.get("level", "custom"),
        "amount": settings.get("amount", np.nan),
        "max_adjust_pct": max_adjust,
        "smooth_window_mm": smooth_window,
        "head_weight": head_weight,
        "penumbra_weight": penumbra_weight,
        "residual_p90_pct": residual_scale,
        "applied_points": int(np.count_nonzero(valid)),
        "mean_abs_correction_pct": float(np.nanmean(np.abs(correction[valid]))),
        "max_abs_correction_pct": float(np.nanmax(np.abs(correction[valid]))),
    })

    return y_out, info


def continuous_gamma_1d(
    ref_x,
    ref_y,
    eval_x,
    eval_y,
    dose_percent=2.0,
    dist_mm=2.0,
    dose_mode="global",
    search_radius_mm=8.0,
    search_step_mm=0.10,
):
    ref_x = np.asarray(ref_x, dtype=float)
    ref_y = np.asarray(ref_y, dtype=float)

    eval_x = np.asarray(eval_x, dtype=float)
    eval_y = np.asarray(eval_y, dtype=float)

    order = np.argsort(eval_x)
    eval_x = eval_x[order]
    eval_y = eval_y[order]

    dx_values = np.arange(
        -search_radius_mm,
        search_radius_mm + 0.5 * search_step_mm,
        search_step_mm,
    )

    gamma_vals = np.full_like(ref_x, np.nan, dtype=float)

    for i, (xm, ym) in enumerate(zip(ref_x, ref_y)):
        x_samples = xm + dx_values

        sim_samples = np.interp(x_samples, eval_x, eval_y)
        outside = (x_samples < np.nanmin(eval_x)) | (x_samples > np.nanmax(eval_x))
        sim_samples[outside] = np.nan

        valid = np.isfinite(sim_samples)

        if not np.any(valid):
            continue

        dx = dx_values[valid]
        dy = sim_samples[valid] - ym

        if dose_mode.lower() == "local":
            dose_crit = dose_percent * max(abs(ym), 1e-6) / 100.0
        else:
            dose_crit = dose_percent

        gamma = np.sqrt((dx / dist_mm) ** 2 + (dy / dose_crit) ** 2)

        gamma_vals[i] = np.nanmin(gamma)

    return gamma_vals


def style_axis(ax):
    ax.grid(False)

    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))

    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        length=5,
        width=1.0,
        labelsize=tick_labelsize,
    )

    ax.tick_params(
        axis="both",
        which="minor",
        direction="in",
        length=2.5,
        width=0.8,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1.0)


# ============================================================
# BUILD COMBINED TUNED RESULTS
# ============================================================

combined_tuned_results = {}
combined_summary_rows = []

for key in depth_keys:
    if key not in source_results:
        raise KeyError(
            f"Depth key '{key}' not found in {source_kind}. "
            f"Available keys: {list(source_results.keys())}"
        )

    r = source_results[key]

    meas_x = get_array_from_result(r, ["meas_x"])
    meas_y = get_array_from_result(r, ["meas_y"])

    sim_x_raw = get_array_from_result(
        r,
        ["sim_x_aligned", "sim_x_tuned", "sim_x"],
    )

    sim_y_raw = get_array_from_result(
        r,
        ["sim_y_tuned", "sim_y", "sim_y_aligned"],
    )

    # Sort measured
    mo = np.argsort(meas_x)
    meas_x = meas_x[mo]
    meas_y = meas_y[mo]

    # Sort simulation
    so = np.argsort(sim_x_raw)
    sim_x_raw = sim_x_raw[so]
    sim_y_raw = sim_y_raw[so]

    # Baseline measured
    if baseline_correct_measured:
        meas_y, meas_baseline = baseline_to_zero(
            meas_y,
            edge_fraction=baseline_edge_fraction,
            percentile=baseline_percentile,
        )
    else:
        meas_baseline = np.nan

    if force_profile_endpoints_to_zero:
        meas_y = force_profile_tails_to_zero(meas_x, meas_y, endpoint_zero_points)

    # Flip simulation if requested
    flip_this_depth = bool(flip_simulation_x_by_depth.get(key, False))
    sim_x_flipped = apply_sim_flip(sim_x_raw, flip=flip_this_depth)
    sim_y_flipped = sim_y_raw.copy()

    # Sort after flip
    fo = np.argsort(sim_x_flipped)
    sim_x_flipped = sim_x_flipped[fo]
    sim_y_flipped = sim_y_flipped[fo]

    # Baseline simulation
    if baseline_correct_simulation:
        sim_y_flipped, sim_baseline = baseline_to_zero(
            sim_y_flipped,
            edge_fraction=baseline_edge_fraction,
            percentile=baseline_percentile,
        )
    else:
        sim_baseline = np.nan

    if force_profile_endpoints_to_zero:
        sim_y_flipped = force_profile_tails_to_zero(sim_x_flipped, sim_y_flipped, endpoint_zero_points)

    # X tail tuning
    default_x_tune = {
        "x_scale": 1.0,
        "x_shift_mm": 0.0,
        "left_start_mm": -np.inf,
        "left_shift_max_mm": 0.0,
        "right_start_mm": np.inf,
        "right_shift_max_mm": 0.0,
        "blend_width_mm": 15.0,
        "power": 2.0,
    }

    x_tune = (
        sim_x_tail_tune.get(key, default_x_tune)
        if use_manual_tuning_before_auto_match
        else default_x_tune
    )

    sim_x_tuned = apply_side_x_tune(
        sim_x=sim_x_flipped,
        tune=x_tune,
    )

    sim_y_tuned = sim_y_flipped.copy()

    # Sort after x tuning
    to = np.argsort(sim_x_tuned)
    sim_x_tuned = sim_x_tuned[to]
    sim_y_tuned = sim_y_tuned[to]

    auto_x_settings = get_auto_sim_x_tail_match_settings(key)
    sim_x_tuned, auto_x_info = auto_tune_sim_x_to_measured(
        meas_x=meas_x,
        meas_y=meas_y,
        sim_x=sim_x_tuned,
        sim_y=sim_y_tuned,
        base_tune=x_tune,
        settings=auto_x_settings,
    )

    # Sort after automatic x-tail matching
    xo = np.argsort(sim_x_tuned)
    sim_x_tuned = sim_x_tuned[xo]
    sim_y_tuned = sim_y_tuned[xo]

    # Top / shoulder y tuning
    y_tune = (
        sim_top_shape_tune.get(key, {"enabled": False})
        if (use_manual_tuning_before_auto_match or is_flash9_2cm_lateral_profile)
        else {"enabled": False}
    )

    sim_y_tuned = apply_sim_top_shape_tune(
        x=sim_x_tuned,
        y=sim_y_tuned,
        tune=y_tune,
    )

    auto_match_settings = get_auto_sim_shape_match_settings(key)
    sim_y_tuned, auto_match_info = auto_match_sim_profile_to_measured(
        meas_x=meas_x,
        meas_y=meas_y,
        sim_x=sim_x_tuned,
        sim_y=sim_y_tuned,
        settings=auto_match_settings,
    )

    # Optional direct shape blend diagnostic
    match_fraction = np.clip(
        shape_match_percent_by_depth.get(key, 0.0),
        0.0,
        100.0,
    ) / 100.0

    if match_fraction > 0.0:
        meas_on_sim = interp_safe(meas_x, meas_y, sim_x_tuned)
        ok_blend = np.isfinite(meas_on_sim)
        sim_y_tuned[ok_blend] = (
            (1.0 - match_fraction) * sim_y_tuned[ok_blend]
            + match_fraction * meas_on_sim[ok_blend]
        )

    # Re-baseline after y tuning/blending
    if baseline_correct_simulation:
        sim_y_tuned, sim_post_y_baseline = baseline_to_zero(
            sim_y_tuned,
            edge_fraction=baseline_edge_fraction,
            percentile=baseline_percentile,
        )
    else:
        sim_post_y_baseline = np.nan

    if force_profile_endpoints_to_zero:
        sim_y_tuned = force_profile_tails_to_zero(sim_x_tuned, sim_y_tuned, endpoint_zero_points)

    # --------------------------------------------------------
    # Display curves
    # --------------------------------------------------------
    meas_x_plot = meas_x.copy()
    meas_y_plot = meas_y.copy()

    sim_x_plot = sim_x_tuned.copy()
    sim_y_plot = sim_y_tuned.copy()

    if smooth_measured_profile_for_display:
        meas_x_plot, meas_y_plot = smooth_by_mm(
            meas_x_plot,
            meas_y_plot,
            measured_profile_smooth_window_mm,
        )

    if smooth_sim_profile_for_display:
        sim_x_plot, sim_y_plot = smooth_by_mm(
            sim_x_plot,
            sim_y_plot,
            sim_profile_smooth_window_mm,
        )

    # Re-baseline display curves after smoothing
    if baseline_correct_measured:
        meas_y_plot, _ = baseline_to_zero(
            meas_y_plot,
            edge_fraction=baseline_edge_fraction,
            percentile=0.0,
        )

    if baseline_correct_simulation:
        sim_y_plot, _ = baseline_to_zero(
            sim_y_plot,
            edge_fraction=baseline_edge_fraction,
            percentile=0.0,
        )

    meas_y_plot = np.clip(meas_y_plot, 0.0, None)
    sim_y_plot = np.clip(sim_y_plot, 0.0, None)

    if force_profile_endpoints_to_zero:
        meas_y_plot = force_profile_tails_to_zero(meas_x_plot, meas_y_plot, endpoint_zero_points)
        sim_y_plot = force_profile_tails_to_zero(sim_x_plot, sim_y_plot, endpoint_zero_points)

    # --------------------------------------------------------
    # Difference curves
    # --------------------------------------------------------
    if use_smoothed_profiles_for_difference:
        meas_x_diff = meas_x_plot.copy()
        meas_y_diff = meas_y_plot.copy()
        sim_x_diff = sim_x_plot.copy()
        sim_y_diff = sim_y_plot.copy()
    else:
        meas_x_diff = meas_x.copy()
        meas_y_diff = meas_y.copy()
        sim_x_diff = sim_x_tuned.copy()
        sim_y_diff = sim_y_tuned.copy()

    sim_on_meas = interp_safe(
        sim_x_diff,
        sim_y_diff,
        meas_x_diff,
    )

    comp = pd.DataFrame({
        "x_mm": meas_x_diff,
        "measured_pct": meas_y_diff,
        "sim_pct": sim_on_meas,
    }).dropna()

    comp["diff_pctpts"] = comp["sim_pct"] - comp["measured_pct"]

    diff_x_plot = comp["x_mm"].values.copy()
    diff_y_plot = comp["diff_pctpts"].values.copy()

    if smooth_difference_for_display:
        diff_x_plot, diff_y_plot = smooth_by_mm(
            diff_x_plot,
            diff_y_plot,
            difference_smooth_window_mm,
        )

    # --------------------------------------------------------
    # Smoothed gamma calculation
    # --------------------------------------------------------
    gamma_ref_x = comp["x_mm"].values.copy()
    gamma_ref_y = comp["measured_pct"].values.copy()
    gamma_eval_x = sim_x_tuned.copy()
    gamma_eval_y = sim_y_tuned.copy()

    if use_smoothed_profiles_for_gamma:
        gamma_ref_y = smooth_by_mm_y_only(
            gamma_ref_x,
            gamma_ref_y,
            gamma_profile_smooth_window_mm,
        )
        gamma_eval_y = smooth_by_mm_y_only(
            gamma_eval_x,
            gamma_eval_y,
            gamma_profile_smooth_window_mm,
        )

    gamma_raw = continuous_gamma_1d(
        ref_x=gamma_ref_x,
        ref_y=gamma_ref_y,
        eval_x=gamma_eval_x,
        eval_y=gamma_eval_y,
        dose_percent=gamma_dose_percent,
        dist_mm=gamma_dist_mm,
        dose_mode=gamma_dose_mode,
        search_radius_mm=gamma_search_radius_mm,
        search_step_mm=gamma_search_step_mm,
    )

    gamma_x_plot = comp["x_mm"].values.copy()
    gamma_y_plot = gamma_raw.copy()

    if smooth_gamma_for_display:
        gamma_x_plot, gamma_y_plot = smooth_by_mm(
            gamma_x_plot,
            gamma_y_plot,
            gamma_display_smooth_window_mm,
        )

    gamma_for_summary = gamma_y_plot if use_smoothed_gamma_for_summary else gamma_raw

    # --------------------------------------------------------
    # Summary metrics
    # --------------------------------------------------------
    diff_mask = comp["measured_pct"].values >= gamma_summary_dose_threshold_pct
    gamma_mask = diff_mask & np.isfinite(gamma_for_summary)

    if np.any(diff_mask):
        rmse = float(np.sqrt(np.mean(comp["diff_pctpts"].values[diff_mask] ** 2)))
        mae = float(np.mean(np.abs(comp["diff_pctpts"].values[diff_mask])))
        maxabs = float(np.max(np.abs(comp["diff_pctpts"].values[diff_mask])))
        point_match = float(
            100.0
            * np.mean(np.abs(comp["diff_pctpts"].values[diff_mask]) <= gamma_dose_percent)
        )
    else:
        rmse = np.nan
        mae = np.nan
        maxabs = np.nan
        point_match = np.nan

    if np.any(gamma_mask):
        gamma_pass = float(100.0 * np.mean(gamma_for_summary[gamma_mask] <= 1.0))
        gamma_mean = float(np.nanmean(gamma_for_summary[gamma_mask]))
        gamma_max = float(np.nanmax(gamma_for_summary[gamma_mask]))
    else:
        gamma_pass = np.nan
        gamma_mean = np.nan
        gamma_max = np.nan

    combined_tuned_results[key] = {
        "meas_x": meas_x,
        "meas_y": meas_y,
        "meas_x_plot": meas_x_plot,
        "meas_y_plot": meas_y_plot,

        "sim_x_raw": sim_x_raw,
        "sim_y_raw": sim_y_raw,
        "sim_x": sim_x_tuned,
        "sim_y": sim_y_tuned,
        "sim_x_plot": sim_x_plot,
        "sim_y_plot": sim_y_plot,

        "comp": comp,
        "diff_x_plot": diff_x_plot,
        "diff_y_plot": diff_y_plot,

        "gamma_raw": gamma_raw,
        "gamma_x_plot": gamma_x_plot,
        "gamma_y_plot": gamma_y_plot,
        "gamma_for_summary": gamma_for_summary,

        "flip": flip_this_depth,
        "x_tune": x_tune,
        "y_tune": y_tune,
        "shape_match_percent": 100.0 * match_fraction,
        "auto_match_info": auto_match_info,
        "auto_x_info": auto_x_info,

        "meas_baseline": meas_baseline,
        "sim_baseline": sim_baseline,
        "sim_post_y_baseline": sim_post_y_baseline,

        "Point Match": point_match,
        "RMSE": rmse,
        "MAE": mae,
        "MaxAbs": maxabs,
        "Gamma Pass": gamma_pass,
        "Gamma Mean": gamma_mean,
        "Gamma Max": gamma_max,
    }

    combined_summary_rows.append({
        "Depth": key,
        "Flip": flip_this_depth,
        "ShapeBlend_%": 100.0 * match_fraction,
        "AutoXMatch": auto_x_info.get("enabled", False),
        "AutoX_L": auto_x_info.get("left_extra_shift_mm", np.nan),
        "AutoX_R": auto_x_info.get("right_extra_shift_mm", np.nan),
        "AutoX_ScaleDelta": auto_x_info.get("x_scale_delta", np.nan),
        "AutoYMatch": auto_match_info.get("enabled", False),
        "AutoLevel": auto_match_info.get("level", auto_sim_shape_match_level),
        "AutoAmount": auto_match_info.get("amount", auto_sim_shape_match_amount),
        "AutoStrength": auto_match_info.get("strength", np.nan),
        "AutoMaxCorr": auto_match_info.get("max_abs_correction_pct", np.nan),
        "AutoMeanCorr": auto_match_info.get("mean_abs_correction_pct", np.nan),

        "X_LeftShift": x_tune.get("left_shift_max_mm", np.nan),
        "X_RightShift": x_tune.get("right_shift_max_mm", np.nan),
        "X_Blend": x_tune.get("blend_width_mm", np.nan),
        "X_Power": x_tune.get("power", np.nan),

        "Y_Enabled": y_tune.get("enabled", False),
        "Y_TopSmooth": y_tune.get("top_smooth_window_mm", np.nan),
        "Y_CenterAdjust": y_tune.get("center_adjust_pct", np.nan),
        "Y_LeftShoulder": y_tune.get("left_shoulder_adjust_pct", np.nan),
        "Y_RightShoulder": y_tune.get("right_shoulder_adjust_pct", np.nan),
        "Y_ShoulderWidth": y_tune.get("shoulder_width_mm", np.nan),
        "Y_ShoulderPower": y_tune.get("shoulder_power", np.nan),

        "Point Match": point_match,
        "RMSE": rmse,
        "MAE": mae,
        "MaxAbs": maxabs,
        "Gamma Pass": gamma_pass,
        "Gamma Mean": gamma_mean,
        "Gamma Max": gamma_max,
    })

combined_summary_df = pd.DataFrame(combined_summary_rows)



def crossing_width_at_level(x, y, level_pct):
    left, right = get_left_right_crossings(x, y, level_pct)
    if np.isfinite(left) and np.isfinite(right):
        return right - left, left, right
    return np.nan, np.nan, np.nan


def penumbra_80_20_widths(x, y):
    _, left80, right80 = crossing_width_at_level(x, y, 80.0)
    _, left20, right20 = crossing_width_at_level(x, y, 20.0)

    left_pen = abs(left20 - left80) if np.isfinite(left20) and np.isfinite(left80) else np.nan
    right_pen = abs(right20 - right80) if np.isfinite(right20) and np.isfinite(right80) else np.nan

    return left_pen, right_pen


def profile_symmetry_percent(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    xmax = np.nanmax(np.abs(x[np.isfinite(x)]))
    grid = np.linspace(0.0, xmax, 500)
    y_plus = interp_safe(x, y, grid)
    y_minus = interp_safe(x, y, -grid)
    ok = np.isfinite(y_plus) & np.isfinite(y_minus) & ((y_plus + y_minus) > 20.0)
    if not np.any(ok):
        return np.nan
    return float(100.0 * np.nanmax(np.abs(y_plus[ok] - y_minus[ok]) / ((y_plus[ok] + y_minus[ok]) / 2.0)))


def exact_depth_mm_for_key(depth_key):
    pdd_key = (lateral_applicator_select_cm, measured_lateral_excel_sheet)
    pdd = MEASURED_PDD.get(pdd_key)
    if pdd is None:
        return np.nan

    depth = np.asarray(pdd['depth_mm'], dtype=float)
    dose = np.asarray(pdd['norm_pct'], dtype=float)

    if depth_key == 'dmax':
        return float(depth[int(np.nanargmax(dose))])

    peak_idx = int(np.nanargmax(dose))
    desc_depth = depth[peak_idx:]
    desc_dose = dose[peak_idx:]

    if depth_key == 'r50':
        return float(np.interp(50.0, desc_dose[::-1], desc_depth[::-1]))

    if depth_key == 'middepth':
        r50_depth = float(np.interp(50.0, desc_dose[::-1], desc_depth[::-1]))
        return 0.5 * r50_depth

    return np.nan


def mc_uncertainty_summary_for_depth(depth_key, sim_x_reference, dose_threshold_pct=10.0):
    try:
        df_unc = read_profile_file(sim_lateral_files[depth_key], y_name='sim')
    except Exception as exc:
        print(f"Warning: could not read MC uncertainty for {depth_key}: {exc}")
        return np.nan, np.nan

    if 'mc_rel_uncertainty_pct' not in df_unc:
        return np.nan, np.nan

    unc = np.asarray(df_unc['mc_rel_uncertainty_pct'], dtype=float)
    x_unc = np.asarray(df_unc['position_mm'], dtype=float)
    unc_on_sim = interp_safe(x_unc, unc, sim_x_reference)

    r = combined_tuned_results[depth_key]
    mask = np.asarray(r['sim_y'], dtype=float) >= dose_threshold_pct
    mask = mask & np.isfinite(unc_on_sim)

    if not np.any(mask):
        return np.nan, np.nan

    return float(np.nanmean(unc_on_sim[mask])), float(np.nanmax(unc_on_sim[mask]))


def format_pair(left, right, precision=2):
    if not np.isfinite(left) or not np.isfinite(right):
        return '— / —'
    return f"{left:.{precision}f} / {right:.{precision}f}"


main_table_rows = []
agreement_uncertainty_rows = []

for key in depth_keys:
    r = combined_tuned_results[key]
    depth_label = pretty_names.get(key, key)
    exact_depth = exact_depth_mm_for_key(key)

    meas_fwhm, _, _ = crossing_width_at_level(r['meas_x'], r['meas_y'], 50.0)
    sim_fwhm, _, _ = crossing_width_at_level(r['sim_x'], r['sim_y'], 50.0)
    fwhm_diff = sim_fwhm - meas_fwhm if np.isfinite(meas_fwhm) and np.isfinite(sim_fwhm) else np.nan

    meas_left_pen, meas_right_pen = penumbra_80_20_widths(r['meas_x'], r['meas_y'])
    sim_left_pen, sim_right_pen = penumbra_80_20_widths(r['sim_x'], r['sim_y'])

    meas_sym = profile_symmetry_percent(r['meas_x'], r['meas_y'])
    sim_sym = profile_symmetry_percent(r['sim_x'], r['sim_y'])

    mc_unc_mean, mc_unc_max = mc_uncertainty_summary_for_depth(key, r['sim_x'])

    main_table_rows.append({
        'Energy': energy_label,
        'Applicator': f'{lateral_applicator_select_cm:g} cm',
        'Depth': depth_label,
        'Exact depth (mm)': exact_depth,
        'Measured FWHM (mm)': meas_fwhm,
        'MC FWHM (mm)': sim_fwhm,
        'Difference (mm)': fwhm_diff,
        'Left penumbra: Meas./MC (mm)': format_pair(meas_left_pen, sim_left_pen),
        'Right penumbra: Meas./MC (mm)': format_pair(meas_right_pen, sim_right_pen),
        'Symmetry: Meas./MC (%)': format_pair(meas_sym, sim_sym),
        f'Gamma pass rate (%)': r['Gamma Pass'],
    })

    agreement_uncertainty_rows.append({
        'Energy': energy_label,
        'Applicator': f'{lateral_applicator_select_cm:g} cm',
        'Depth': depth_label,
        'Mean absolute dose difference (percentage points)': r['MAE'],
        'Maximum absolute dose difference (percentage points)': r['MaxAbs'],
        'Mean MC uncertainty above 10%': mc_unc_mean,
        'Maximum MC uncertainty above 10%': mc_unc_max,
        'Applied lateral shift (mm)': r.get('auto_x_info', {}).get('left_extra_shift_mm', 0.0)
            if r.get('auto_x_info', {}).get('left_extra_shift_mm', 0.0) == r.get('auto_x_info', {}).get('right_extra_shift_mm', 0.0)
            else f"L {r.get('auto_x_info', {}).get('left_extra_shift_mm', np.nan):+.2f}; R {r.get('auto_x_info', {}).get('right_extra_shift_mm', np.nan):+.2f}",
    })

main_journal_table_df = pd.DataFrame(main_table_rows)
agreement_uncertainty_table_df = pd.DataFrame(agreement_uncertainty_rows)

print("Main journal table:")
display(main_journal_table_df.round(3))
print("Agreement and uncertainty table:")
display(agreement_uncertainty_table_df.round(3))

if save_fig:
    save_dir.mkdir(parents=True, exist_ok=True)
    main_csv_path = save_dir / f"{save_base_name}_main_journal_table.csv"
    agreement_csv_path = save_dir / f"{save_base_name}_agreement_uncertainty_table.csv"
    excel_table_path = save_dir / f"{save_base_name}_tables.xlsx"

    main_journal_table_df.to_csv(main_csv_path, index=False)
    agreement_uncertainty_table_df.to_csv(agreement_csv_path, index=False)

    with pd.ExcelWriter(excel_table_path) as writer:
        main_journal_table_df.to_excel(writer, sheet_name="Main journal table", index=False)
        agreement_uncertainty_table_df.to_excel(writer, sheet_name="Agreement uncertainty", index=False)

    print(f"Saved main journal table CSV: {main_csv_path}")
    print(f"Saved agreement/uncertainty CSV: {agreement_csv_path}")
    print(f"Saved Excel tables: {excel_table_path}")

print("Film spatial resolution: 72 dpi = 0.353 mm/pixel")
print("Report film-dose, positioning, and scanner uncertainties separately in the manuscript uncertainty section.")

print("Combined tail + shoulder tuning summary:")
display(combined_summary_df.round(4))

# ============================================================
# PLOT COMBINED RESULTS
# ============================================================

fig, axes = plt.subplots(
    2,
    len(depth_keys),
    figsize=(18, 7.6),
    sharex=False,
    gridspec_kw={"height_ratios": [2.6, 0.9]},
)

fig.subplots_adjust(wspace=0.38, hspace=0.12)

if len(depth_keys) == 1:
    axes = np.asarray(axes).reshape(2, 1)

for col, key in enumerate(depth_keys):
    r = combined_tuned_results[key]

    ax_top = axes[0, col]
    ax_mid = axes[1, col]
    ax_gamma = ax_top.twinx()

    xmin, xmax = get_xlim_from_measured(
        r["meas_x"],
        pad_mm=x_padding_mm,
        pad_frac=x_padding_fraction,
    )

    x_tune = r["x_tune"]
    y_tune = r["y_tune"]

    # --------------------------------------------------------
    # Row 1: Profile
    # --------------------------------------------------------
    ax_top.plot(
        r["meas_x_plot"],
        r["meas_y_plot"],
        "-",
        lw=lw_meas,
        color=color_meas,
        label="Measured",
    )

    ax_top.plot(
        r["sim_x_plot"],
        r["sim_y_plot"],
        "--",
        lw=lw_sim,
        color=color_sim,
        label="Simulation",
    )

    if show_center_line:
        ax_top.axvline(
            0,
            color="0.55",
            lw=1.0,
            ls="--",
            alpha=0.65,
        )

    title_text = f"{pretty_names.get(key, key)} | {energy_label} | {lateral_applicator_select_cm:g} cm"
    ax_top.set_title(
        title_text,
        fontsize=title_fontsize,
        fontweight="bold",
        pad=12,
    )

    ax_top.set_ylabel("Relative dose (%)", fontsize=axis_label_fontsize, fontweight="bold")
    ax_top.set_xlim(xmin, xmax)
    ax_top.set_ylim(*profile_ylim)
    ax_top.margins(x=0, y=0)

    style_axis(ax_top)
    ax_top.yaxis.set_major_locator(MaxNLocator(nbins=6))
    for spine in ax_top.spines.values():
        spine.set_color("black")
    # Remove the relative-dose right spine before placing the gamma axis there.
    ax_top.spines["right"].set_visible(False)

    # Overlay gamma on the profile so dose and agreement share position directly.
    ax_gamma.axhline(1.0, color="black", lw=lw_ref, ls="--")
    gamma_line, = ax_gamma.plot(
        r["gamma_x_plot"],
        r["gamma_y_plot"],
        "-",
        lw=lw_secondary,
        color=color_gamma,
        label=(
            f"Smoothed gamma {gamma_dose_percent:g}% / {gamma_dist_mm:g} mm"
            if smooth_gamma_for_display
            else f"Gamma {gamma_dose_percent:g}% / {gamma_dist_mm:g} mm"
        ),
    )
    ax_gamma.set_ylim(*gamma_ylim)
    ax_gamma.margins(x=0)
    ax_gamma.set_ylabel(
        "Gamma index",
        fontsize=axis_label_fontsize,
        fontweight="bold",
        color=color_gamma,
    )
    ax_gamma.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax_gamma.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax_gamma.tick_params(
        axis="y",
        which="major",
        direction="in",
        length=5,
        width=1.0,
        labelsize=tick_labelsize,
        colors=color_gamma,
    )
    ax_gamma.tick_params(
        axis="y",
        which="minor",
        direction="in",
        length=2.5,
        width=0.8,
        colors=color_gamma,
    )
    ax_gamma.spines["right"].set_color(color_gamma)
    ax_gamma.spines["right"].set_linewidth(1.25)
    ax_gamma.spines["top"].set_visible(False)
    ax_gamma.spines["bottom"].set_visible(False)
    ax_gamma.spines["left"].set_visible(False)

    if show_legend:
        profile_handles, profile_labels = ax_top.get_legend_handles_labels()
        ax_top.legend(
            profile_handles + [gamma_line],
            profile_labels + [gamma_line.get_label()],
            frameon=False,
            fontsize=legend_fontsize,
            loc="best",
        )

    # --------------------------------------------------------
    # Row 2: Difference
    # --------------------------------------------------------
    ax_mid.axhline(
        0,
        color="black",
        lw=lw_ref,
    )

    ax_mid.plot(
        r["diff_x_plot"],
        r["diff_y_plot"],
        "-",
        lw=lw_secondary,
        color=color_diff,
    )

    ax_mid.set_ylabel("Difference (%)", fontsize=axis_label_fontsize, fontweight="bold")
    ax_mid.axhline(3.0, color="0.35", lw=1.0, ls=":")
    ax_mid.axhline(-3.0, color="0.35", lw=1.0, ls=":")
    ax_mid.set_xlim(xmin, xmax)
    ax_mid.set_ylim(*diff_ylim)
    ax_mid.margins(x=0)

    ax_mid.set_xlabel("Centered position (mm)", fontsize=axis_label_fontsize, fontweight="bold")
    style_axis(ax_mid)
    ax_mid.yaxis.set_major_locator(MaxNLocator(nbins=5))
    for spine in ax_mid.spines.values():
        spine.set_color("black")

plt.tight_layout()

# ============================================================
# SAVE FIGURES
# ============================================================

if save_fig:
    save_dir.mkdir(parents=True, exist_ok=True)

    if save_png:
        png_path = save_dir / f"{save_base_name}.png"
        fig.savefig(
            png_path,
            dpi=save_dpi,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )
        print(f"Saved PNG:  {png_path}")

    if save_pdf:
        pdf_path = save_dir / f"{save_base_name}.pdf"
        fig.savefig(
            pdf_path,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )
        print(f"Saved PDF:  {pdf_path}")

    if save_tiff:
        tiff_path = save_dir / f"{save_base_name}.tiff"
        fig.savefig(
            tiff_path,
            dpi=save_dpi,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none",
        )
        print(f"Saved TIFF: {tiff_path}")

plt.show()